# Participant installation guide

> Complete this setup before the workshop. Standard Google Colab is not supported.

## macOS

### 1. Install Apple command-line tools

This provides utilities such as `make` and Git:

```bash
xcode-select --install
```

If already installed, macOS will report that no installation is necessary.

### 2. Install Homebrew

Skip this if `brew --version` already works.

```bash
/bin/bash -c "$(curl -fsSL https://raw.githubusercontent.com/Homebrew/install/HEAD/install.sh)"
```

Follow Homebrew’s printed instructions for adding `brew` to your shell, then reopen Terminal.

### 3. Install and start Docker Desktop

```bash
brew install --cask docker
open -a Docker
```

Wait for Docker Desktop to finish starting, then verify:

```bash
docker info
```

Docker is ready only when `docker info` completes without a daemon connection error.

### 4. Install Python

```bash
brew install python
```

### 5. Install kind

```bash
brew install kind
```

### 6. Install the workshop-compatible kubectl version

```bash
brew install kubectl
kubectl version --client
```

### 7. Install Jupyter inside the workshop directory

```bash
cd "kubesummit/tutorial/nvlink-topology-workshop 2"

python3 -m venv .venv
source .venv/bin/activate

python -m pip install --upgrade pip
python -m pip install jupyterlab
```

### 8. Verify and prepare

```bash
docker info
kind version
kubectl version --client
python3 --version
make --version
curl --version

make preflight
make prepare
```

### 9. Start the notebook

```bash
source .venv/bin/activate
jupyter lab workshop-walkthrough.ipynb
```

---

## Windows 10/11

The workshop must run inside WSL2. Native Command Prompt and PowerShell are not supported for the lab commands.

### 1. Install WSL2

Open PowerShell as Administrator:

```powershell
wsl --install -d Ubuntu
wsl --update
wsl --set-default-version 2
```

Restart Windows when requested and complete the Ubuntu username/password setup.

### 2. Install Docker Desktop

In Administrator PowerShell:

```powershell
winget install --exact --id Docker.DockerDesktop
```

Restart Windows if requested, then open Docker Desktop.

In Docker Desktop:

1. Open **Settings**.
2. Select **Resources → WSL Integration**.
3. Enable integration for Ubuntu.
4. Apply the changes and restart Docker Desktop.

### 3. Install Linux dependencies inside Ubuntu/WSL

Open the Ubuntu terminal—not PowerShell:

```bash
sudo apt-get update

sudo apt-get install -y \
  bash \
  make \
  curl \
  ca-certificates \
  python3 \
  python3-pip \
  python3-venv
```

Verify that WSL can reach Docker Desktop:

```bash
docker info
```

Do not continue until this succeeds.

### 4. Install kind inside WSL

```bash
KIND_VERSION=v0.30.0

case "$(dpkg --print-architecture)" in
  amd64) KIND_ARCH=amd64 ;;
  arm64) KIND_ARCH=arm64 ;;
  *) echo "Unsupported CPU architecture"; exit 1 ;;
esac

curl -fLo kind \
  "https://kind.sigs.k8s.io/dl/${KIND_VERSION}/kind-linux-${KIND_ARCH}"

chmod +x kind
sudo install kind /usr/local/bin/kind
rm kind
```

### 5. Install kubectl inside WSL

```bash
KUBECTL_VERSION=v1.34.0

case "$(dpkg --print-architecture)" in
  amd64) KUBECTL_ARCH=amd64 ;;
  arm64) KUBECTL_ARCH=arm64 ;;
  *) echo "Unsupported CPU architecture"; exit 1 ;;
esac

curl -fLo kubectl \
  "https://dl.k8s.io/release/${KUBECTL_VERSION}/bin/linux/${KUBECTL_ARCH}/kubectl"

curl -fLo kubectl.sha256 \
  "https://dl.k8s.io/release/${KUBECTL_VERSION}/bin/linux/${KUBECTL_ARCH}/kubectl.sha256"

echo "$(cat kubectl.sha256)  kubectl" | sha256sum --check

chmod +x kubectl
sudo install kubectl /usr/local/bin/kubectl
rm kubectl kubectl.sha256
```

### 6. Install Jupyter inside the workshop directory

Run inside Ubuntu/WSL:

```bash
cd "kubesummit/tutorial/nvlink-topology-workshop 2"

python3 -m venv .venv
source .venv/bin/activate

python -m pip install --upgrade pip
python -m pip install jupyterlab
```

### 7. Verify and prepare

```bash
docker info
kind version
kubectl version --client
python3 --version
make --version
curl --version

make preflight
make prepare
```

### 8. Start Jupyter from WSL

```bash
source .venv/bin/activate
jupyter lab --no-browser workshop-walkthrough.ipynb
```

Jupyter prints a URL beginning with `http://localhost:8888`. Open that URL in the Windows browser.

---

## Expected preparation result

`make prepare` should finish with:

```text
✓ READY FOR KUBESUMMIT
```

Participants should complete this before the event. During the workshop, begin with:

```bash
make checkpoint-1
```

# The Scheduler Has Never Heard of NVLink
### Topology-Aware GPU Placement on Kubernetes — advanced attendee workbook

This notebook is the executable companion to the **39-slide live deck**. The deck establishes the mental model; the notebook proves the scheduling behavior and exposes the implementation.

**Environment:** Bash on macOS/Linux or WSL2, Docker, kind, kubectl, curl, make, and Python 3. **No physical GPU is required.** Fake extended resources model Kubernetes capacity; they do not model CUDA, NVLink bandwidth, NCCL, DCGM, or training performance.

Every experiment follows the same contract:

> **QUESTION → START STATE → ACTION → EXPECTED EVIDENCE → INTERPRETATION → RECOVERY**

Run the notebook from the repository root. Do not paste commands from screenshots; the Makefile and versioned manifests are the source of truth.

## Complete workshop story

### Phase 0 — Prepare the laptop

```bash
make preflight
make prepare
```

We verify that Docker, kind, kubectl and the other tools are available. We also download the required images and Kueue installation manifest before the workshop.

Nothing is scheduled yet.

---

### Phase 1 — Build a miniature GPU datacenter

```bash
make checkpoint-1
```

We create a local Kubernetes cluster containing:

```text
                         block-1
               ┌────────────┴────────────┐
             leaf-a                   leaf-b
       ┌────────┴────────┐       ┌────────┴────────┐
     worker           worker2   worker3          worker4
      1 GPU             1 GPU    1 GPU             1 GPU
```

These are not real GPUs. We advertise a fake extended resource:

```text
workshop.example.com/gpu
```

We also add labels describing where each worker is located:

```text
block → leaf → hostname
```

Kubernetes can now see four GPU units, but its default scheduler does not understand that the GPUs belong to different network domains.

---

### Phase 2 — Demonstrate the default scheduler’s limitation

```bash
make lab1-demo
```

We want to create a predictable bad placement.

First, we place two filler Pods:

```text
worker  in leaf-a → occupied
worker3 in leaf-b → occupied
```

The remaining free workers are:

```text
worker2 → leaf-a
worker4 → leaf-b
```

We then submit two trainer Pods requesting one GPU each.

Because the Job does not declare any relationship between its trainers, Kubernetes schedules them independently:

```text
trainer-0 → worker2 → leaf-a
trainer-1 → worker4 → leaf-b
```

Both Pods are running, all resource requests are satisfied, and Kubernetes reports no error.

The placement is technically valid but potentially inefficient for distributed training because communication must cross leaf-switch domains.

---

### Phase 3 — Reset the experiment

```bash
make lab1-reset
```

The trainer and filler Pods run `agnhost pause`, so they do not finish themselves.

We delete them and wait until all four fake GPUs are free again. This ensures that the next experiment starts from a clean state.

---

### Phase 4 — Install Kueue and describe topology-aware capacity

```bash
make checkpoint-2
```

We install Kueue and create four connected objects:

```text
Topology/gpu-fabric
        ↓
ResourceFlavor/h100-topology
        ↓
ClusterQueue/gpu-training
        ↓
LocalQueue/gpu-queue
```

Their responsibilities are:

- **Topology:** Defines `block → leaf → hostname`.
- **ResourceFlavor:** Identifies which workers provide the GPU capacity.
- **ClusterQueue:** Provides quota for four fake GPUs.
- **LocalQueue:** Gives Jobs in the `training` namespace a queue to use.

Before this phase, Kubernetes saw four unrelated GPUs.

After this phase, Kueue understands:

```text
leaf-a contains two GPU units
leaf-b contains two GPU units
block-1 contains all four GPU units
```

No training Job is running yet. We have only prepared the topology-aware control plane.

---

### Phase 5 — Require both trainers to use one leaf

```bash
make lab3-required
```

We submit a two-replica training Job containing:

```yaml
kueue.x-k8s.io/podset-required-topology:
  workshop.example.com/leaf
```

This means:

> “Do not start this Job unless both trainer replicas can fit inside one leaf.”

Kueue evaluates the complete PodSet:

```text
2 trainers × 1 GPU = 2 GPUs required
```

Both leaves have two free GPUs, so Kueue selects one of them.

For example:

```text
trainer-0 → worker  → leaf-a
trainer-1 → worker2 → leaf-a
```

Kueue reserves both GPU units together and admits the complete Job. Kubernetes’ scheduler then performs the final Pod-to-Node binding.

This proves successful topology-aware admission and placement.

---

### Phase 6 — Deliberately request an impossible topology

```bash
make lab3-impossible
```

We now request:

```text
3 trainers × 1 GPU
all required inside one leaf
```

But the topology contains:

```text
leaf-a → only 2 GPUs
leaf-b → only 2 GPUs
```

The cluster has four GPUs in total, so aggregate capacity is sufficient. The problem is that no individual leaf has three GPUs.

Kueue therefore produces:

```text
QuotaReserved=False
Trainer Pods=0
```

It does not start two trainers while leaving the third pending. That would consume GPUs without allowing synchronous distributed training to progress.

The waiting Workload is the correct result because the same-leaf requirement is a hard promise.

---

### Phase 7 — Change required locality to preferred locality

```bash
make lab3-preferred
```

The Job still requests three GPUs, but the policy changes from:

```text
required topology
```

to:

```text
preferred topology
```

Kueue first tries to place all three replicas in one leaf. That is impossible, so it widens the search to the parent domain, `block-1`.

The Job can now run across both leaves:

```text
trainer-0 → leaf-a
trainer-1 → leaf-a
trainer-2 → leaf-b
```

This demonstrates an explicit business trade-off:

- **Required:** Preserve locality, even if the Job must wait.
- **Preferred:** Try locality first, but allow wider placement to reduce queue time.

---

### Phase 8 — Restore the clean topology-aware state

```bash
make checkpoint-3
```

We remove the experimental Jobs, Workloads, trainers and filler Pods.

We preserve:

- The kind cluster.
- Four worker Nodes.
- Four fake GPUs.
- Node topology labels.
- Kueue controller.
- Topology and ResourceFlavor.
- ClusterQueue and LocalQueue.

The cluster is clean but remains ready to repeat any experiment.

---

### Phase 9 — Connect the lab to a production GPU platform

Finally, we discuss what a real implementation requires:

- Obtain topology from LLDP, DCIM, NetBox and fabric-management APIs.
- Normalize and validate that information.
- Publish trustworthy Node labels.
- Use Kueue TAS for cluster-level placement.
- Use Kubernetes Topology Manager for GPU, NIC, CPU and NUMA alignment inside each Node.
- Validate workload topology policies at admission time.
- Observe communication and GPU efficiency using DCGM and MFU.
- Benchmark same-leaf, cross-leaf and cross-rack placements.

## The simple takeaway

```text
First:
Kubernetes sees four identical GPUs and schedules Pods independently.

Then:
Kueue learns how those GPUs are physically related.

Finally:
Kueue admits the complete training Job according to required or preferred
topology policy before Kubernetes schedules its Pods.
```

The workshop is fundamentally about moving from:

> “Wherever a GPU is free”

to:

> “Where the complete distributed Job can run efficiently and according to policy.”

## Start here — establish the notebook working directory

The “repository root” for this workbook means the directory containing this workshop's `Makefile`, `jobs/`, `kueue-config/`, and `scripts/`:

```bash
cd "kubesummit/tutorial/nvlink-topology-workshop 2"
make preflight
make prepare
jupyter lab workshop-walkthrough.ipynb
```

If the workshop folder was copied elsewhere, `cd` to that copy instead.

Why this matters: every `%%bash` cell is a new shell process, but it inherits the Jupyter server's current directory. A `cd` performed inside one Bash cell does **not** carry into the next cell. Launching Jupyter here makes every relative path deterministic.

### How to read the notebook

Every runnable cell begins with a specific action heading such as **Create the lab cluster**, **Start the filler and trainer Pods**, or **Prove strict locality**.

- The short explanation, action, and expected result remain visible.
- Small badges tell you whether the step is live, optional, read-only, or mutating.
- Detailed implementation, state changes, interpretation, and recovery are inside a collapsed section. Expand it while preparing or troubleshooting; leave it closed while presenting.

The Makefile is the stable participant interface; the guide exposes its implementation so convenience does not become opacity.

### Confirm that Jupyter is in the workshop directory

`REFERENCE`  `READ-ONLY`

**Why this cell is here**

Fail immediately if Jupyter was launched from the wrong directory instead of discovering the problem halfway through a lab.

**What it does**

- Resolves the kernel's current directory.
- Checks for five workshop landmarks used by later cells.

**Expected result**

The output shows `WORKSHOP ROOT: .../nvlink-topology-workshop 2` followed by `OK`.

<details>
<summary><strong>Implementation details, interpretation and recovery</strong></summary>

**Runs or reads**

- Python's standard-library `pathlib`; it does not invoke Docker or Kubernetes.

**State changed:** None. This is a local filesystem inspection.

**How to interpret the result:** Passing proves paths such as `jobs/...` and `scripts/...` will resolve. It says nothing yet about Docker or the cluster.

**If it fails:** Stop the Jupyter server, `cd` to the workshop directory, relaunch it, and rerun this cell. Changing directory in one `%%bash` cell is not a durable fix.

</details>

In [143]:
from pathlib import Path

workshop_root = Path.cwd().resolve()
required = ["Makefile", "scripts", "jobs", "kueue-config", "cluster-setup"]
missing = [name for name in required if not (workshop_root / name).exists()]
assert not missing, (
    f"Wrong working directory: {workshop_root}\n"
    f"Missing: {', '.join(missing)}\n"
    "Restart Jupyter from the nvlink-topology-workshop 2 directory."
)
print(f"WORKSHOP ROOT: {workshop_root}")
print("OK — relative paths in later cells will resolve correctly.")


WORKSHOP ROOT: /Users/neeraj/Desktop/devopsconfRU/kubesummit/tutorial/nvlink-topology-workshop 2
OK — relative paths in later cells will resolve correctly.


## 90-minute route

| Time | Live slides | Activity | Primary medium |
|---:|---:|---|---|
| 0–12 min | 1–8 | Problem, communication hierarchy, ownership | Slides |
| 12–27 min | 9–14 | Phase 1: fake datacenter and topology-blind placement | Notebook |
| 27–42 min | 15–20 | Phase 2: model the fabric in Kueue | Slides + notebook |
| 42–64 min | 21–28 | Phase 3: required, impossible, preferred locality | Notebook |
| 64–73 min | 29–31 | Gang scheduling, TAS, Cohort, Topology Manager | Slides + workbook reference |
| 73–82 min | 32–37 | Inventory pipeline, policy, evidence | Notebook |
| 82–87 min | 38–39 | Production architecture and rules | Slides |
| 87–90 min | — | Questions or recovery buffer | — |

The live labs are the center of the workshop. Advanced production sections are designed for guided inspection rather than installation on the fake cluster.

## 1 — The physical model before Kubernetes

> **Presentation mapping — Live slides 1–8:** use this section while explaining the topology problem, bandwidth hierarchy, AllReduce, and responsibility map. No live cluster action is required yet.

A distributed training job does not consume an unordered bag of GPUs. It consumes positions in a communication graph:

```text
GPU ↔ NVLink/NVSwitch ↔ GPU
          │
       PCIe root / NUMA
          │
      ConnectX NIC
          │
   leaf switch ↔ spine ↔ leaf switch
```

Reference orders of magnitude:

| Link | Advertised rate | Scope |
|---|---:|---|
| H100 NVLink | approximately 900 GB/s aggregate | GPU-to-GPU inside the node |
| PCIe Gen5 x16 | approximately 64 GB/s per direction | device-to-host/NIC path |
| InfiniBand NDR | 400 Gb/s ≈ 50 GB/s raw line rate | node-to-node |
| InfiniBand HDR | 200 Gb/s ≈ 25 GB/s raw line rate | prior-generation node-to-node |

The units matter: **Gb/s is gigabits; GB/s is gigabytes**. Real application bandwidth is lower than raw line rate.

### Why AllReduce makes placement visible

Data-parallel training repeatedly executes:

```text
forward → backward → gradient AllReduce → optimizer step
```

Communication time depends on message size, collective algorithm, link bandwidth, topology, and congestion. NCCL can optimize within the placement it receives; it cannot move a Pod to another Kubernetes node.

This laptop lab proves:

- labels can describe topology domains;
- Kueue can admit a whole PodSet into a domain;
- required locality can wait instead of partially allocating GPUs;
- preferred locality can widen under pressure.

It does **not** prove bandwidth, MFU, NCCL efficiency, or a universal 40–60% performance difference.

## 2 — Preflight before conference Wi-Fi

> **Presentation mapping — before Live slide 1:** complete this preparation before the session. If necessary, show only the final readiness result before beginning the deck.

Preparation has two deliberately separate stages. `make preflight` is read-only. `make prepare` uses the network and writes only to the workshop cache plus Docker Desktop's image store.

### Check that this laptop can run the workshop

`PRE-EVENT`  `READ-ONLY`

**Why this cell is here**

Prove that the host has the command-line tools and minimum host-side capacity needed by the workshop.

**What it does**

- Locates Bash, Docker, kind, kubectl, curl, make, and Python 3.
- Calls `docker info` to prove the Docker daemon is reachable.
- Uses `df` on the workshop filesystem and requires at least 5 GiB of host free space.
- On Windows, requires WSL2/Bash.

**Expected result**

Every line begins with `OK`, followed by `CHECKS PASSED — run 'make prepare'...`.

<details>
<summary><strong>Implementation details, interpretation and recovery</strong></summary>

**Runs or reads**

- `Makefile` target `preflight`.
- `scripts/preflight.sh --check`.

**State changed:** None. It does not pull images, create a cluster, switch context, or modify Kubernetes.

**How to interpret the result:** This validates host prerequisites only. It cannot guarantee that Docker Desktop's internal Linux disk has enough free space for five kind nodes.

**If it fails:** Fix each line marked `MISSING`; start Docker Desktop if the daemon is unavailable, then rerun the cell.

**Safety / caveat:** Host disk and Docker Desktop VM disk are different capacity pools. A later `no space left on device` from containerd means Docker's internal storage needs cleanup or a larger disk allocation even if this check passed.

</details>

In [144]:
%%bash
make preflight


bash scripts/preflight.sh --check
OK  bash -> /bin/bash
OK  docker -> /usr/local/bin/docker
OK  kind -> /opt/homebrew/bin/kind
OK  kubectl -> /opt/homebrew/bin/kubectl
OK  curl -> /usr/bin/curl
OK  make -> /usr/bin/make
OK  python3 -> /usr/bin/python3
OK  Docker daemon reachable
OK  at least 5 GiB free disk
CHECKS PASSED — run 'make prepare' while you still have reliable internet.


### Download and cache the workshop dependencies

`PRE-EVENT`  `NEEDS INTERNET`

**Why this cell is here**

Make the live checkpoints independent of conference Wi-Fi by downloading every pinned dependency in advance.

**What it does**

- Repeats all preflight checks.
- Creates `.workshop-cache/`.
- Downloads Kueue `v0.19.2` to a temporary file, checks that it contains a Deployment, then atomically renames it to `.workshop-cache/kueue-v0.19.2.yaml`.
- Pulls `kindest/node:v1.34.0`, `registry.k8s.io/e2e-test-images/agnhost:2.53`, and every controller image discovered from the pinned Kueue manifest.

**Expected result**

The final line is `✓ READY FOR KUBESUMMIT` and lists the pinned kind image, lab image, and Kueue version.

<details>
<summary><strong>Implementation details, interpretation and recovery</strong></summary>

**Runs or reads**

- `Makefile` target `prepare`.
- `scripts/preflight.sh --prepare`.
- GitHub Releases through `curl` and image registries through `docker pull`.

**State changed:** Writes one manifest beneath `.workshop-cache/` and adds images to Docker Desktop's image store. It does not create the kind cluster.

**How to interpret the result:** The host now owns the installation inputs. Later checkpoints copy these cached images into kind; they do not pull from the network.

**If it fails:** Keep reliable internet available, free host/Docker storage if needed, and rerun `make prepare`. The temporary-download/atomic-move design prevents a partial manifest from replacing a valid cache.

</details>

In [ ]:
%%bash
make prepare


### Confirm that the offline cache is ready

`PRE-EVENT`  `READ-ONLY`

**Why this cell is here**

Verify the artifacts that `make prepare` promised instead of treating its readiness banner as a black box.

**What it does**

- Confirms the manifest is non-empty and contains a Deployment.
- Prints immutable local image IDs.
- Attempts to summarize Docker images, containers, volumes, and build cache; this optional report is handled as a warning if Docker's cache metadata is unhealthy.

**Expected result**

Three image IDs print without errors. A Docker usage table normally follows; a clearly labeled warning is acceptable because it does not invalidate the required cached artifacts.

<details>
<summary><strong>Implementation details, interpretation and recovery</strong></summary>

**Runs or reads**

- POSIX file tests and `grep` for the cached manifest.
- `docker image inspect` for each pinned image.
- `docker system df` for Docker-managed usage and reclaimable space.

**State changed:** None.

**How to interpret the result:** The manifest and three image inspections are the readiness assertions. `docker system df` is diagnostics only. Images live inside Docker Desktop's Linux VM/disk image, not this repository.

**If it fails:** Rerun `make prepare` for a missing manifest/image. If only `docker system df` fails with a missing `overlay2` path, restart Docker Desktop. If it persists, `docker builder prune` removes unused build cache after confirmation; avoid broad `docker system prune --volumes` because it can delete unrelated data.

</details>

In [ ]:
%%bash
test -s .workshop-cache/kueue-v0.19.2.yaml
grep -m1 'kind: Deployment' .workshop-cache/kueue-v0.19.2.yaml
docker image inspect kindest/node:v1.34.0 --format 'kind image: {{.Id}}'
docker image inspect registry.k8s.io/e2e-test-images/agnhost:2.53 --format 'lab image:  {{.Id}}'
docker image inspect registry.k8s.io/kueue/kueue:v0.19.2 --format 'Kueue:     {{.Id}}'
if ! docker system df; then
  echo "WARN: required workshop artifacts passed, but Docker's optional usage report failed."
  echo "Restart Docker Desktop; if it persists, inspect/reclaim unused build cache with: docker builder prune"
fi


### Validate the workshop repository

`PRE-EVENT`  `READ-ONLY`

**Why this cell is here**

Catch broken workshop assets without depending on Docker, a Kubernetes cluster, or conference networking.

**What it does**

- Runs `bash -n` on every shell entry point.
- Dry-runs important Make recipes.
- Parses every YAML document, validates the inventory against JSON Schema, validates both notebooks, and checks that their Bash cells parse.
- Tests the topology renderer, admission-policy core, and benchmark summarizer.

**Expected result**

`Static validation passed...` followed by all Python tests reporting `OK`.

<details>
<summary><strong>Implementation details, interpretation and recovery</strong></summary>

**Runs or reads**

- `tests/static-test.sh`.
- Python `unittest` discovery for `tests/test_*.py`.

**State changed:** None. `make test` is not the end-to-end cluster test; that is the optional destructive `make smoke`.

**How to interpret the result:** The repository is internally coherent. It does not prove that Docker has enough internal disk or that a live kind cluster can start.

**If it fails:** Read the first failing file/test in the output. Do not continue to the live labs until this target passes.

</details>

In [ ]:
%%bash
make test


### Recovery contract

| Symptom | Recovery command | Restored state |
|---|---|---|
| Cluster, labels, or fake resources drifted | `make checkpoint-1` | Five Ready nodes and four allocatable fake GPUs |
| Lab 1 Pods still own capacity | `make lab1-reset` | All four fake GPUs free |
| Kueue or TAS objects are missing | `make checkpoint-2` | Controller available and ClusterQueue Active |
| Lab 3 Workloads collide | `make checkpoint-3` | TAS ready with zero lab workloads |

In a 40-person room, use checkpoints instead of debugging one laptop serially.

---
# Phase 1 — Build a fake datacenter and reproduce bad placement

> **Presentation mapping — Live slides 9–14:** build the fake topology, reproduce cross-leaf placement, inspect the result, and reset the capacity boundary.

The cluster has one control-plane node and four worker nodes. Each worker advertises one fake GPU.

```text
                       block-1
              ┌──────────┴──────────┐
            leaf-a                leaf-b
       ┌──────┴──────┐       ┌──────┴──────┐
     worker       worker2   worker3       worker4
      1 GPU         1 GPU    1 GPU          1 GPU
```

The resource name is `workshop.example.com/gpu` so nobody mistakes it for a real NVIDIA device plugin resource.

## 3 — Restore Checkpoint 1

> **Presentation cue — Live slides 9–10:** establish the four-GPU, two-leaf baseline before explaining the transaction boundary.

This idempotent command creates the named kind cluster if needed, loads cached images, applies the topology inventory, advertises fake resources, removes stale workloads, and verifies the result.

### Create or restore the four-GPU fake datacenter

`LIVE`  `CHANGES LAB`

**Why this cell is here**

Converge every laptop on the same five-node, four-fake-GPU starting state for slides 9–10.

**What it does**

- Creates cluster `topology-lab` only if absent; otherwise reuses it.
- Selects context `kind-topology-lab`, loads pre-cached images into all kind nodes, and applies the `training` namespace.
- Validates inventory, labels four workers as two leaves, patches one fake GPU into each worker's Node status, waits for allocatable, deletes stale lab workloads, then asserts the complete baseline.

**Expected result**

Ends with `CHECKPOINT 1 VERIFIED` and `CHECKPOINT 1 — fake GPU datacenter ready`; five Nodes are Ready and four workers each expose one allocatable fake GPU.

<details>
<summary><strong>Implementation details, interpretation and recovery</strong></summary>

**Runs or reads**

- `Makefile` → `scripts/checkpoint.sh 1`.
- That script composes kind, `load-images.sh`, namespace YAML, topology rendering/labeling, fake-GPU advertisement, workload reset, and `verify-lab1.sh`.

**State changed:** May create five Docker containers for kind; switches the current kubectl context; labels/patches Nodes; removes workshop-labeled Jobs and Pods. It never touches another Kubernetes context.

**How to interpret the result:** The simulated infrastructure facts now exist in the Kubernetes API. No Kueue controller or topology-aware policy is installed yet.

**If it fails:** It is idempotent: rerun the same target. If image loading says `no space left on device`, delete the partial lab cluster with `make clean`, reclaim/increase Docker Desktop storage, then rerun `make checkpoint-1`.

</details>

In [148]:
%%bash
make checkpoint-1


# Under the hood:
# Makefile
#   └─ scripts/checkpoint.sh 1
#        ├─ kind create cluster
#        │    └─ cluster-setup/kind-config.yaml
#        ├─ kubectl apply namespaces
#        ├─ cluster-setup/label-topology.sh
#        │    ├─ reads sample-inventory.json
#        │    ├─ validates through render-topology.py
#        │    └─ runs kubectl label node
#        ├─ cluster-setup/advertise-fake-gpus.sh
#        └─ scripts/verify-lab1.sh



# label-topology.sh executes:

# kubectl label node topology-lab-worker \
#   workshop.example.com/block=block-1 \
#   workshop.example.com/leaf=leaf-a \
#   workshop.example.com/rack=rack-a1 \
#   --overwrite

# kubectl label node topology-lab-worker2 \
#   workshop.example.com/block=block-1 \
#   workshop.example.com/leaf=leaf-a \
#   workshop.example.com/rack=rack-a1 \
#   --overwrite

# kubectl label node topology-lab-worker3 \
#   workshop.example.com/block=block-1 \
#   workshop.example.com/leaf=leaf-b \
#   workshop.example.com/rack=rack-b1 \
#   --overwrite

# kubectl label node topology-lab-worker4 \
#   workshop.example.com/block=block-1 \
#   workshop.example.com/leaf=leaf-b \
#   workshop.example.com/rack=rack-b1 \
#   --overwrite

bash scripts/checkpoint.sh 1
==> Loading registry.k8s.io/e2e-test-images/agnhost:2.53 into topology-lab


Image: "registry.k8s.io/e2e-test-images/agnhost:2.53" with ID "sha256:6debb3645a281a2a03eb859d3fb46a9baf27218aaf817d3e95505db1ec170f8d" found to be already present on all nodes.


==> Loading registry.k8s.io/kueue/kueue:v0.19.2 into topology-lab


Image: "registry.k8s.io/kueue/kueue:v0.19.2" with ID "sha256:656064ccfc30ea3a0eed8ef34a50a96652c89c30cc33ce93f7639c7be3458ba9" found to be already present on all nodes.


namespace/training unchanged
OK: 4 nodes satisfy topology inventory v1
==> Labeling topology-lab-worker -> block=block-1 leaf=leaf-a rack=rack-a1
node/topology-lab-worker not labeled
==> Labeling topology-lab-worker2 -> block=block-1 leaf=leaf-a rack=rack-a1
node/topology-lab-worker2 not labeled
==> Labeling topology-lab-worker3 -> block=block-1 leaf=leaf-b rack=rack-b1
node/topology-lab-worker3 not labeled
==> Labeling topology-lab-worker4 -> block=block-1 leaf=leaf-b rack=rack-b1
node/topology-lab-worker4 not labeled
NAME                         STATUS   ROLES           AGE   VERSION   BLOCK     LEAF     RACK
topology-lab-control-plane   Ready    control-plane   10m   v1.34.0                      
topology-lab-worker          Ready    <none>          10m   v1.34.0   block-1   leaf-a   rack-a1
topology-lab-worker2         Ready    <none>          10m   v1.34.0   block-1   leaf-a   rack-a1
topology-lab-worker3         Ready    <none>          10m   v1.34.0   block-1   leaf-b   rack-b1


In [149]:
%%bash
kubectl get nodes \
  -L workshop.example.com/block \
  -L workshop.example.com/leaf \
  -L workshop.example.com/rack

NAME                         STATUS   ROLES           AGE   VERSION   BLOCK     LEAF     RACK
topology-lab-control-plane   Ready    control-plane   10m   v1.34.0                      
topology-lab-worker          Ready    <none>          10m   v1.34.0   block-1   leaf-a   rack-a1
topology-lab-worker2         Ready    <none>          10m   v1.34.0   block-1   leaf-a   rack-a1
topology-lab-worker3         Ready    <none>          10m   v1.34.0   block-1   leaf-b   rack-b1
topology-lab-worker4         Ready    <none>          10m   v1.34.0   block-1   leaf-b   rack-b1


## 4 — Inspect the topology source contract

> **Presentation cue — Live slide 9; revisited on slides 32–33:** connect the visual topology to the versioned inventory that produces the Node labels.

Node labels should be output from an owned data pipeline, not handwritten folklore. The sample inventory contains:

- normalized block, leaf, rack, NIC, and GPU SKU;
- provenance for rack, leaf, GPU, and NUMA facts;
- a schema version and generation timestamp.

### Field dictionary

| Field | Plain-language meaning | Example in this lab |
|---|---|---|
| `$schema` | Points editors and tooling to the local validation/documentation contract | `./inventory.schema.json` |
| `schemaVersion` | Version of our inventory contract—not the Kubernetes version | `v1` |
| `generatedAt` | When this normalized snapshot was produced; used to detect stale data | `2026-09-02T00:00:00Z` |
| `name` | Exact Kubernetes Node name that will receive the labels | `topology-lab-worker2` |
| `block` | Coarse network/fabric domain containing multiple leaf switches | `block-1` |
| `leaf` | Leaf-switch domain connected to the node; our important locality boundary | `leaf-a` |
| `rack` | Physical rack location | `rack-a1` |
| `nic` | High-speed network-interface identifier | `mlx5_0` |
| `gpuSku` | Normalized accelerator model/capacity class; `h100-sxm5` means NVIDIA H100 SXM5-class | `h100-sxm5` |
| `sources` | Provenance: which authoritative system supplied each fact | NetBox, LLDP/fabric API, GPU Operator, kubelet |

The values describe the production-style contract we want to teach. The kind workers still have **fake extended GPU resources**, not physical H100s or ConnectX NICs.

In production, these facts normally come from DCIM, LLDP/fabric APIs, GPU Operator/DRA, and kubelet/device discovery.

### Read the datacenter map and its validation rules

`REFERENCE`  `READ-ONLY`

**Why this cell is here**

The first file is our map of the fake datacenter. The second file is the rulebook that says what a valid map must contain. This cell only displays both files.

**What it does**

- Prints the normalized node records: name, block, leaf, rack, NIC, GPU SKU, NUMA data, and field provenance.
- Prints the JSON Schema that constrains types and required fields.

**Expected result**

Four worker records appear, with two mapped to `leaf-a` and two to `leaf-b`; the schema identifies required inventory fields.

<details>
<summary><strong>Implementation details, interpretation and recovery</strong></summary>

**Runs or reads**

- `sed` against `topology-pipeline/sample-inventory.json` and `inventory.schema.json`.

**State changed:** None; both files are only read.

**How to interpret the result:** The inventory is infrastructure data, while the schema is its interface contract. Neither file by itself mutates Kubernetes.

**If it fails:** If a path is missing, rerun the working-directory guard. If output is truncated in the notebook UI, open the files directly.

</details>

In [150]:
%%bash
sed -n '1,220p' topology-pipeline/sample-inventory.json
sed -n '1,240p' topology-pipeline/inventory.schema.json


{
  "$schema": "./inventory.schema.json",
  "schemaVersion": "v1",
  "generatedAt": "2026-09-02T00:00:00Z",
  "nodes": [
    {
      "name": "topology-lab-worker",
      "block": "block-1",
      "leaf": "leaf-a",
      "rack": "rack-a1",
      "nic": "mlx5_0",
      "gpuSku": "h100-sxm5",
      "sources": {"rack": "netbox", "leaf": "lldp+fabric-api", "gpuSku": "gpu-operator", "numa": "kubelet"}
    },
    {
      "name": "topology-lab-worker2",
      "block": "block-1",
      "leaf": "leaf-a",
      "rack": "rack-a1",
      "nic": "mlx5_0",
      "gpuSku": "h100-sxm5",
      "sources": {"rack": "netbox", "leaf": "lldp+fabric-api", "gpuSku": "gpu-operator", "numa": "kubelet"}
    },
    {
      "name": "topology-lab-worker3",
      "block": "block-1",
      "leaf": "leaf-b",
      "rack": "rack-b1",
      "nic": "mlx5_0",
      "gpuSku": "h100-sxm5",
      "sources": {"rack": "netbox", "leaf": "lldp+fabric-api", "gpuSku": "gpu-operator", "numa": "kubelet"}
    },
    {
      "name": "t

### Validate the inventory and render a readable topology table

`LIVE`  `READ-ONLY`

**Why this cell is here**

Demonstrate the validation and normalization boundary between infrastructure inventory and Kubernetes publication.

**What it does**

- Parses JSON, checks schema version, non-empty/unique node names, normalized label syntax, and provenance for rack/leaf/GPU/NUMA fields.
- Sorts nodes and renders a deterministic table.

**Expected result**

`OK: 4 nodes satisfy topology inventory v1`, followed by a four-row table.

<details>
<summary><strong>Implementation details, interpretation and recovery</strong></summary>

**Runs or reads**

- `scripts/render-topology.py` first in validation mode, then in human-readable table mode.

**State changed:** None. `--check` and `--format table` do not label Nodes.

**How to interpret the result:** Only validated, deterministic topology should enter the scheduling control plane. The publisher later consumes the same renderer as TSV.

**If it fails:** Read the `ERROR:` line and correct the named inventory field. Do not bypass validation and publish guessed labels.

</details>

In [151]:
%%bash
python3 scripts/render-topology.py --check
python3 scripts/render-topology.py --format table


OK: 4 nodes satisfy topology inventory v1
NODE	BLOCK	LEAF	RACK	LEAF SOURCE
topology-lab-worker	block-1	leaf-a	rack-a1	lldp+fabric-api
topology-lab-worker2	block-1	leaf-a	rack-a1	lldp+fabric-api
topology-lab-worker3	block-1	leaf-b	rack-b1	lldp+fabric-api
topology-lab-worker4	block-1	leaf-b	rack-b1	lldp+fabric-api


### Confirm that the topology labels reached Kubernetes

`LIVE`  `READ-ONLY`

**Why this cell is here**

Compare the intended inventory with the labels actually published on Kubernetes Nodes.

**What it does**

- Queries the current cluster and joins each Node's Ready/role information with block, leaf, and rack labels.

**Expected result**

All five Nodes appear; the four workers show `block-1`, an even `leaf-a`/`leaf-b` split, and distinct rack labels. The control-plane may have blank workshop labels.

<details>
<summary><strong>Implementation details, interpretation and recovery</strong></summary>

**Runs or reads**

- `kubectl get nodes` with three label columns.

**State changed:** None.

**How to interpret the result:** Scheduler-visible topology is Node metadata. This is the published view, not proof that LLDP/DCIM facts are correct in the physical world.

**If it fails:** Confirm context with `kubectl config current-context`; if labels are missing, rerun `make checkpoint-1`.

</details>

In [152]:
%%bash
kubectl get nodes -L workshop.example.com/block -L workshop.example.com/leaf -L workshop.example.com/rack


NAME                         STATUS   ROLES           AGE   VERSION   BLOCK     LEAF     RACK
topology-lab-control-plane   Ready    control-plane   10m   v1.34.0                      
topology-lab-worker          Ready    <none>          10m   v1.34.0   block-1   leaf-a   rack-a1
topology-lab-worker2         Ready    <none>          10m   v1.34.0   block-1   leaf-a   rack-a1
topology-lab-worker3         Ready    <none>          10m   v1.34.0   block-1   leaf-b   rack-b1
topology-lab-worker4         Ready    <none>          10m   v1.34.0   block-1   leaf-b   rack-b1


## 5 — Understand the fake extended resource

> **Presentation cue — supports Live slides 9–10:** explain how the lab creates scheduler-visible capacity. This implementation detail does not need a full projected walkthrough.

The lab patches the Node `status` subresource because extended-resource capacity is reported as node status. The script then waits for `status.allocatable`: scheduler feasibility uses **allocatable**, not merely capacity.

This is a simulation mechanism for kind. A production cluster uses a device plugin or DRA driver; operators should not patch GPU capacity manually.

### See how the lab creates fake GPU capacity

`OPTIONAL`  `READ-ONLY`

**Why this cell is here**

This displays the script that makes each ordinary kind worker appear to Kubernetes as if it owns one GPU. It shows the script; it does not run it again.

**What it does**

- Shows a JSON Patch to `/status/capacity/workshop.example.com~1gpu` for each deterministic worker name.
- Shows the subsequent polling of `status.allocatable`, the field the scheduler actually consumes.

**Expected result**

You can identify the status-subresource patch and `wait_for_gpu_allocatable` loop.

<details>
<summary><strong>Implementation details, interpretation and recovery</strong></summary>

**Runs or reads**

- `sed` against `cluster-setup/advertise-fake-gpus.sh`.

**State changed:** None; this cell prints the script. The script itself already ran inside Checkpoint 1.

**How to interpret the result:** This is a control-plane simulation. It does not provide a device, driver, CUDA runtime, NVLink, or isolation.

**If it fails:** If the file is missing, check the working directory. Do not run the patch pattern against a production node.

</details>

In [153]:
%%bash
sed -n '1,220p' cluster-setup/advertise-fake-gpus.sh


#!/usr/bin/env bash
# PURPOSE: Give each kind worker one scheduler-visible *fake* GPU.
# CALLED BY: scripts/checkpoint.sh after topology labels are published.
# MUTATES: Node/status only in kind-topology-lab.
# PRODUCTION DIFFERENCE: Real GPUs are advertised by a device plugin or DRA
# driver. This script provides accounting semantics but no device, CUDA, NVLink,
# health monitoring, or isolation and must never be copied to production.
# SUCCESS: Every worker reaches capacity=1 and allocatable=1.
set -euo pipefail  # Stop on an error, an unset variable, or a failed pipeline.

# Resolve the workshop root from this script's location, so the repository can
# be cloned into any filesystem directory.
ROOT=$(cd "$(dirname "$0")/.." && pwd)

# Load shared constants (GPU_RESOURCE, cluster name) and helper functions.
. "$ROOT/scripts/lib.sh"

# Refuse to patch Nodes unless kubectl points at kind-topology-lab.
require_lab_context

# JSON Pointer escapes `/` in the resource name as `~1`.
# This a

### Confirm one fake GPU is schedulable on every worker

`LIVE`  `READ-ONLY`

**Why this cell is here**

Inspect capacity and allocatable separately so the fake resource is not mistaken for merely decorative metadata.

**What it does**

- Reads both `.status.capacity` and `.status.allocatable` for every Node.

**Expected result**

Each worker reports `capacity=1 allocatable=1`; the control-plane reports empty values.

<details>
<summary><strong>Implementation details, interpretation and recovery</strong></summary>

**Runs or reads**

- `kubectl get nodes` with a Go template indexing the extended-resource key.

**State changed:** None.

**How to interpret the result:** Capacity describes what the node owns. Allocatable is the schedulable amount after reservation; kube-scheduler filters on allocatable.

**If it fails:** Rerun `make checkpoint-1`. If capacity appears before allocatable, the checkpoint waits up to 90 seconds for kubelet reconciliation.

</details>

In [154]:
%%bash
kubectl get nodes -o go-template='{{range .items}}{{.metadata.name}}{{"  capacity="}}{{index .status.capacity "workshop.example.com/gpu"}}{{"  allocatable="}}{{index .status.allocatable "workshop.example.com/gpu"}}{{"\n"}}{{end}}'


topology-lab-control-plane  capacity=<no value>  allocatable=<no value>
topology-lab-worker  capacity=1  allocatable=1
topology-lab-worker2  capacity=1  allocatable=1
topology-lab-worker3  capacity=1  allocatable=1
topology-lab-worker4  capacity=1  allocatable=1


## 6 — Lab 1 hypothesis: quantity without locality

> **Presentation mapping — Live slides 11–14:** inspect the unaware Job, run the deterministic cross-leaf case, interpret the evidence, and reset.

The default scheduler can satisfy two independent one-GPU Pods without knowing that they are peer ranks in one collective. To make the demonstration deterministic, filler Pods occupy one GPU in each leaf. The remaining feasible nodes are on different leaves.

Before executing, inspect the intent:

- filler Pods are pinned to known nodes;
- the training Job requests one fake GPU per replica;
- the Job has no Kueue queue label and no topology annotation.

### The two files have different jobs

1. **`placement-fillers.yaml` prepares the experiment.** It creates `leaf-a-filler` on `worker` and `leaf-b-filler` on `worker3`. Each holds one fake GPU, leaving only `worker2` and `worker4` available.
2. **`training-unaware.yaml` is the workload under test.** It creates a Job whose controller creates two trainer Pods. Each requests one fake GPU, but neither asks to share a leaf.

The following inspection cell creates nothing. The Pods appear only when we later run `make lab1-demo`.

### Meet the two kinds of Pods used in Lab 1

`LIVE`  `READ-ONLY`

**Why this cell is here**

`placement-fillers.yaml` creates two harmless placeholder Pods that deliberately occupy one fake GPU in each leaf. `training-unaware.yaml` defines a Kubernetes Job whose controller creates the two pretend trainer Pods we want to observe. This cell only displays the files; `make lab1-demo` applies them later.

**What it does**

- Shows two filler Pods pinned to workers in different leaves, each consuming one fake GPU.
- Shows a two-replica Job requesting one fake GPU per Pod, with no queue label or TAS annotation.

**Expected result**

You can point to each filler's hostname `nodeSelector`, `parallelism: 2` on the Job, the extended-resource request/limit, and the absence of Kueue topology intent.

<details>
<summary><strong>Implementation details, interpretation and recovery</strong></summary>

**Runs or reads**

- `sed` against the filler Pod manifest and the topology-unaware Job.

**State changed:** None.

**How to interpret the result:** The scheduler sees four independent resource consumers. Nothing declares that the two trainer Pods exchange gradients or should share a leaf.

**If it fails:** If the manifests differ from this description, stop and run `make test`; the checked-in assets may have drifted.

</details>

In [155]:
%%bash
sed -n '1,180p' jobs/placement-fillers.yaml
sed -n '1,200p' jobs/training-unaware.yaml


# PURPOSE: Make Lab 1's topology-blind result deterministic on every laptop.
# USED BY: `make lab1-demo`; deleted by `make lab1-reset`.
# PRECONDITION: Checkpoint 1 has published one fake GPU on each worker.
# ACTION: Occupy worker (leaf-a) and worker3 (leaf-b), one GPU each.
# RESULT: Only worker2 (leaf-a) and worker4 (leaf-b) remain feasible for the
# two topology-unaware trainer Pods, forcing a demonstrable cross-leaf result.
# IMPORTANT: These are capacity holders, not simulated training processes.
apiVersion: v1
kind: Pod
metadata:
  name: leaf-a-filler
  namespace: training
  labels:
    workshop.example.com/role: placement-filler
spec:
  # Direct pinning is intentional here: this Pod creates experiment conditions.
  nodeSelector:
    kubernetes.io/hostname: topology-lab-worker
  automountServiceAccountToken: false
  terminationGracePeriodSeconds: 1
  securityContext:
    seccompProfile:
      type: RuntimeDefault
  containers:
  - name: holder
    image: registry.k8s.io/e2e-test

### Create two filler Pods, then start two trainer Pods

`LIVE`  `CHANGES LAB`

**Why this cell is here**

Create a deterministic example where default scheduling is valid by resource quantity but poor for communication locality.

**What it does**

- Clears previous lab workloads and waits for GPU release.
- Starts one filler in each leaf, waits for them to become Ready, then creates the two-replica unaware Job.
- Waits for both trainers and resolves their Node labels, failing unless two distinct leaves are observed.

**Expected result**

Ends with `default scheduler produced a valid but topology-unaware cross-leaf placement` and prints one trainer node from each leaf.

<details>
<summary><strong>Implementation details, interpretation and recovery</strong></summary>

**Runs or reads**

- `scripts/reset-workloads.sh`.
- `jobs/placement-fillers.yaml` and `jobs/training-unaware.yaml`.
- `scripts/verify-unaware-placement.sh`.

**State changed:** Creates two filler Pods, one Job, and two trainer Pods in namespace `training`; all four fake GPUs become allocated.

**How to interpret the result:** Kubernetes did not make an illegal decision; the workload omitted locality semantics. Deterministic fillers make the missing contract visible.

**If it fails:** Run `make lab1-reset`, then `make checkpoint-1`, and rerun this cell. Inspect Pending Pods with `kubectl describe pod -n training <name>` if the verifier times out.

</details>

In [156]:
%%bash
make lab1-demo


# Under the hood:
# bash scripts/reset-workloads.sh

# kubectl apply -f jobs/placement-fillers.yaml
# kubectl wait pod \
#   -n training \
#   -l workshop.example.com/role=placement-filler \
#   --for=condition=Ready \
#   --timeout=90s

# kubectl apply -f jobs/training-unaware.yaml

# bash scripts/verify-unaware-placement.sh

bash scripts/reset-workloads.sh
==> Deleting workshop Jobs and deterministic filler Pods
No resources found
No resources found
OK  all four fake GPUs are available for the next exercise
kubectl apply -f jobs/placement-fillers.yaml
pod/leaf-a-filler created
pod/leaf-b-filler created
kubectl wait pod -n training -l workshop.example.com/role=placement-filler --for=condition=Ready --timeout=90s
pod/leaf-a-filler condition met
pod/leaf-b-filler condition met
kubectl apply -f jobs/training-unaware.yaml
job.batch/training-unaware created
bash scripts/verify-unaware-placement.sh
pod/training-unaware-89cmb condition met
pod/training-unaware-jkz6x condition met
topology-lab-worker2             -> leaf-a
topology-lab-worker4             -> leaf-b
OK  default scheduler produced a valid but topology-unaware cross-leaf placement


### Show where the trainer Pods actually landed

`LIVE`  `READ-ONLY`

**Why this cell is here**

Trace the result from Job declaration to Pod-to-Node binding and then to physical leaf labels.

**What it does**

- Shows each trainer's `NODE`.
- Maps those node names to leaf domains.
- Shows Job status and recent scheduling-related events.

**Expected result**

Two Ready trainer Pods appear on nodes whose leaf columns differ; Job desired/current counts are two.

<details>
<summary><strong>Implementation details, interpretation and recovery</strong></summary>

**Runs or reads**

- Three `kubectl` reads: trainer Pods, labeled Nodes, and the Job description/events.

**State changed:** None.

**How to interpret the result:** Pod phase proves execution; Node plus leaf proves physical placement. Neither alone proves model throughput.

**If it fails:** If Pods are Pending, inspect the Events section and confirm the fillers and fake GPU allocatable state. Restore with `make checkpoint-1` if state is unclear.

</details>

In [157]:
%%bash
kubectl get pods -n training -l job-name=training-unaware -o wide
kubectl get nodes -L workshop.example.com/leaf
kubectl describe job training-unaware -n training | sed -n '1,120p'


NAME                     READY   STATUS    RESTARTS   AGE   IP           NODE                   NOMINATED NODE   READINESS GATES
training-unaware-89cmb   1/1     Running   0          2s    10.244.4.2   topology-lab-worker2   <none>           <none>
training-unaware-jkz6x   1/1     Running   0          2s    10.244.2.2   topology-lab-worker4   <none>           <none>
NAME                         STATUS   ROLES           AGE   VERSION   LEAF
topology-lab-control-plane   Ready    control-plane   10m   v1.34.0   
topology-lab-worker          Ready    <none>          10m   v1.34.0   leaf-a
topology-lab-worker2         Ready    <none>          10m   v1.34.0   leaf-a
topology-lab-worker3         Ready    <none>          10m   v1.34.0   leaf-b
topology-lab-worker4         Ready    <none>          10m   v1.34.0   leaf-b
Name:             training-unaware
Namespace:        training
Selector:         batch.kubernetes.io/controller-uid=2e29da64-9f9b-4775-a350-e85c67d3b3cb
Labels:           worksho

### Interpretation checkpoint

Answer before continuing:

1. Did Kubernetes violate any declared constraint? **No.**
2. Did it know these Pods were communication peers? **No.**
3. Would `topologySpreadConstraints` solve packing? Usually not; it primarily expresses spreading/skew.
4. Would plain `nodeAffinity` be enough? Only if the submitter already knew which currently available leaf could fit the entire group.

The failure is not an incorrect scheduler decision. It is an incomplete workload contract.

### Delete the Lab 1 Pods and verify a clean baseline

`LIVE`  `CHANGES LAB`

**Why this cell is here**

Release all four fake GPUs and prove the infrastructure baseline survived Lab 1.

**What it does**

- Deletes workshop-labeled Jobs and filler Pods with server-side waiting.
- Polls until matching Pods are actually gone, then rechecks node count, topology labels, and allocatable fake GPUs.

**Expected result**

Reports `all four fake GPUs are available` and `CHECKPOINT 1 VERIFIED`.

<details>
<summary><strong>Implementation details, interpretation and recovery</strong></summary>

**Runs or reads**

- `scripts/reset-workloads.sh`, then `scripts/verify-lab1.sh`.

**State changed:** Deletes only workshop workloads in namespace `training`; it preserves the cluster, labels, and capacity declaration.

**How to interpret the result:** Deletion is asynchronous, so the explicit wait creates a trustworthy capacity boundary for Phase 2.

**If it fails:** Rerun the cell. If objects are stuck terminating, inspect them before forcing deletion; the safe broad recovery is `make checkpoint-1`.

</details>

In [158]:
%%bash
make lab1-demo

bash scripts/reset-workloads.sh
==> Deleting workshop Jobs and deterministic filler Pods
job.batch "training-unaware" deleted from training namespace
pod "leaf-a-filler" deleted from training namespace
pod "leaf-b-filler" deleted from training namespace
OK  all four fake GPUs are available for the next exercise
kubectl apply -f jobs/placement-fillers.yaml
pod/leaf-a-filler created
pod/leaf-b-filler created
kubectl wait pod -n training -l workshop.example.com/role=placement-filler --for=condition=Ready --timeout=90s
pod/leaf-a-filler condition met
pod/leaf-b-filler condition met
kubectl apply -f jobs/training-unaware.yaml
job.batch/training-unaware created
bash scripts/verify-unaware-placement.sh
pod/training-unaware-gkzp4 condition met
pod/training-unaware-nxmsv condition met
topology-lab-worker2             -> leaf-a
topology-lab-worker4             -> leaf-b
OK  default scheduler produced a valid but topology-unaware cross-leaf placement


In [159]:
%%bash
make lab1-reset
make verify-lab1


# Makefile
# ├─ target: lab1-reset
# │  └─ scripts/reset-workloads.sh
# │     ├─ sources scripts/lib.sh
# │     │  ├─ defines cluster context: kind-topology-lab
# │     │  ├─ defines namespace: training
# │     │  └─ provides bounded waiting functions
# │     ├─ verifies current kubectl context
# │     ├─ deletes workshop Jobs
# │     │  └─ Kubernetes also deletes their trainer Pods
# │     ├─ deletes the two filler Pods
# │     ├─ waits until trainer Pods are gone
# │     └─ waits until filler Pods are gone
# │
# └─ target: verify-lab1
#    └─ scripts/verify-lab1.sh
#       ├─ sources scripts/lib.sh
#       ├─ verifies current kubectl context
#       ├─ verifies exactly 5 Nodes exist
#       ├─ verifies all 5 Nodes are Ready
#       ├─ verifies worker → block-1 / leaf-a
#       ├─ verifies worker2 → block-1 / leaf-a
#       ├─ verifies worker3 → block-1 / leaf-b
#       ├─ verifies worker4 → block-1 / leaf-b
#       ├─ verifies every worker has allocatable GPU=1
#       ├─ verifies zero trainer Pods remain
#       └─ verifies zero filler Pods remain


bash scripts/reset-workloads.sh
==> Deleting workshop Jobs and deterministic filler Pods
job.batch "training-unaware" deleted from training namespace
pod "leaf-a-filler" deleted from training namespace
pod "leaf-b-filler" deleted from training namespace
OK  all four fake GPUs are available for the next exercise
bash scripts/verify-lab1.sh
OK  5 nodes present
OK  all 5 nodes Ready
OK  topology-lab-worker -> block-1/leaf-a, allocatable GPU=1
OK  topology-lab-worker2 -> block-1/leaf-a, allocatable GPU=1
OK  topology-lab-worker3 -> block-1/leaf-b, allocatable GPU=1
OK  topology-lab-worker4 -> block-1/leaf-b, allocatable GPU=1
OK  zero workshop workload Pods; all four fake GPUs are free
OK  CHECKPOINT 1 VERIFIED — deterministic four-GPU topology ready


---
# Phase 2 - Represent topology in Kueue

> **Presentation mapping — Live slides 15–20:** explain admission versus placement, read the four-object contract, then install and verify it with Checkpoint 2.

Kueue is an admission controller in front of kube-scheduler:

```text
Job → LocalQueue → ClusterQueue quota + ResourceFlavor
                                  │
                                  └→ Topology hierarchy
                                             ↓
                                Workload topologyAssignment
                                             ↓
                                  kube-scheduler binds Nodes
```

Kueue chooses an admissible domain and reserves capacity for the PodSet. The Kubernetes scheduler still performs final Pod-to-node binding.

## 7 — Restore Checkpoint 2

> **Presentation cue — Live slide 20:** run this after slides 15–19 have explained the Kueue object chain.

This installs pinned Kueue assets from the local cache, marks eligible nodes, applies all four Kueue objects, and waits for an Active ClusterQueue.

### Install Kueue and create the topology-aware queues

`LIVE`  `CHANGES LAB`

**Why this cell is here**

Layer Kueue Topology Aware Scheduling (TAS) on the verified fake datacenter.

**What it does**

- Re-establishes clean topology/capacity state.
- Server-side applies the cached Kueue `v0.19.2` release and waits up to 300 seconds for its controller Deployment.
- Marks GPU workers flavor-eligible; applies Topology, ResourceFlavor, ClusterQueue, and LocalQueue; asserts links, hierarchy, controller availability, and `Active=True`.

**Expected result**

Ends with `CHECKPOINT 2 VERIFIED — Kueue topology-aware admission ready` and `CHECKPOINT 2 — Kueue + TAS ready`.

<details>
<summary><strong>Implementation details, interpretation and recovery</strong></summary>

**Runs or reads**

- `scripts/checkpoint.sh 2`, which first runs all of Checkpoint 1.
- `scripts/install-kueue.sh`, `label-gpu-nodes.sh`, four manifests under `kueue-config/`, and `verify-lab2.sh`.

**State changed:** Installs Kueue CRDs/controller into `kueue-system`, labels four worker Nodes, and creates queue/TAS API objects. It may update existing objects but does not duplicate them.

**How to interpret the result:** The admission plane can now reserve a complete PodSet in a topology domain. kube-scheduler remains responsible for final Node binding.

**If it fails:** Rerun `make checkpoint-2`. If the controller is unavailable, inspect `kubectl get pods -n kueue-system` and `kubectl describe deployment kueue-controller-manager -n kueue-system`. A missing cache requires `make prepare`.

</details>

In [160]:
%%bash
make checkpoint-2

# make checkpoint-2 prepares the cluster for the Kueue topology-aware scheduling exercises.


# Phase 1: restore the simulated GPU cluster
# scripts/checkpoint.sh 2
# ├─ Creates topology-lab if missing
# │  └─ cluster-setup/kind-config.yaml
# ├─ Selects kubectl context kind-topology-lab
# ├─ Loads cached workshop images
# ├─ Creates/reconciles namespace training
# │  └─ cluster-setup/namespaces.yaml
# ├─ Publishes block, leaf and rack labels
# │  └─ cluster-setup/label-topology.sh
# ├─ Advertises one fake GPU on each worker
# │  └─ cluster-setup/advertise-fake-gpus.sh
# ├─ Deletes previous trainer and filler Pods
# │  └─ scripts/reset-workloads.sh
# └─ Verifies the five-node fake GPU cluster
#    └─ scripts/verify-lab1.sh



#  Phase 2
# - kueue install : scripts/install-kueue.sh
# - Reads the cached manifest for the versions: .workshop-cache/kueue-v0.19.2.yaml
# - Installs the Kueue CRD's, permissions, Service and controller : kubectl apply --server-side -f "$KUEUE_MANIFEST"

# - Waits upto 5 mins: 
#   kubectl wait deployment/kueue-controller-manager \
  # -n kueue-system \
  # --for=condition=Available \
  # --timeout=300s

# - Mark the GPU workers: cluster-setup/label-gpu-nodes.sh
# - Label the 4 workers: workshop.example.com/gpu-node: "true" == That identifies which Nodes can supply the h100-topology ResourceFlavor.
# - Applies the kueue object chain:
# kueue-config/topology.yaml
#         ↓
# kueue-config/resource-flavor.yaml
#         ↓
# kueue-config/cluster-queue.yaml
#         ↓
# kueue-config/local-queue.yaml


# This creates:
# Topology/gpu-fabric
#     Defines block → leaf → hostname

# ResourceFlavor/h100-topology
#     Selects GPU workers and references gpu-fabric

# ClusterQueue/gpu-training
#     Provides a quota of four fake GPUs

# training/LocalQueue/gpu-queue
#     Gives training Jobs a namespaced queue


# Finally, scripts/verify-lab2.sh checks:
# - Kueue controller has an available replica.
# - gpu-fabric contains the expected topology levels.
# - h100-topology selects GPU workers.
# - h100-topology references gpu-fabric.
# - gpu-training reports Active=True.
# - training/gpu-queue points to gpu-training.

bash scripts/checkpoint.sh 2
==> Loading registry.k8s.io/e2e-test-images/agnhost:2.53 into topology-lab


Image: "registry.k8s.io/e2e-test-images/agnhost:2.53" with ID "sha256:6debb3645a281a2a03eb859d3fb46a9baf27218aaf817d3e95505db1ec170f8d" found to be already present on all nodes.


==> Loading registry.k8s.io/kueue/kueue:v0.19.2 into topology-lab


Image: "registry.k8s.io/kueue/kueue:v0.19.2" with ID "sha256:656064ccfc30ea3a0eed8ef34a50a96652c89c30cc33ce93f7639c7be3458ba9" found to be already present on all nodes.


namespace/training unchanged
OK: 4 nodes satisfy topology inventory v1
==> Labeling topology-lab-worker -> block=block-1 leaf=leaf-a rack=rack-a1
node/topology-lab-worker not labeled
==> Labeling topology-lab-worker2 -> block=block-1 leaf=leaf-a rack=rack-a1
node/topology-lab-worker2 not labeled
==> Labeling topology-lab-worker3 -> block=block-1 leaf=leaf-b rack=rack-b1
node/topology-lab-worker3 not labeled
==> Labeling topology-lab-worker4 -> block=block-1 leaf=leaf-b rack=rack-b1
node/topology-lab-worker4 not labeled
NAME                         STATUS   ROLES           AGE   VERSION   BLOCK     LEAF     RACK
topology-lab-control-plane   Ready    control-plane   10m   v1.34.0                      
topology-lab-worker          Ready    <none>          10m   v1.34.0   block-1   leaf-a   rack-a1
topology-lab-worker2         Ready    <none>          10m   v1.34.0   block-1   leaf-a   rack-a1
topology-lab-worker3         Ready    <none>          10m   v1.34.0   block-1   leaf-b   rack-b1


customresourcedefinition.apiextensions.k8s.io/admissionchecks.kueue.x-k8s.io serverside-applied


customresourcedefinition.apiextensions.k8s.io/clusterqueues.kueue.x-k8s.io serverside-applied
customresourcedefinition.apiextensions.k8s.io/cohorts.kueue.x-k8s.io serverside-applied
customresourcedefinition.apiextensions.k8s.io/localqueues.kueue.x-k8s.io serverside-applied
customresourcedefinition.apiextensions.k8s.io/multikueueclusters.kueue.x-k8s.io serverside-applied
customresourcedefinition.apiextensions.k8s.io/multikueueconfigs.kueue.x-k8s.io serverside-applied
customresourcedefinition.apiextensions.k8s.io/provisioningrequestconfigs.kueue.x-k8s.io serverside-applied
customresourcedefinition.apiextensions.k8s.io/resourceflavors.kueue.x-k8s.io serverside-applied
customresourcedefinition.apiextensions.k8s.io/topologies.kueue.x-k8s.io serverside-applied
customresourcedefinition.apiextensions.k8s.io/workloadpriorityclasses.kueue.x-k8s.io serverside-applied
customresourcedefinition.apiextensions.k8s.io/workloads.kueue.x-k8s.io serverside-applied
serviceaccount/kueue-controller-manager s

Error from server (InternalError): error when creating "/Users/neeraj/Desktop/devopsconfRU/kubesummit/tutorial/nvlink-topology-workshop 2/kueue-config/resource-flavor.yaml": Internal error occurred: failed calling webhook "mresourceflavor.kb.io": failed to call webhook: Post "https://kueue-webhook-service.kueue-system.svc:443/mutate-kueue-x-k8s-io-v1beta2-resourceflavor?timeout=10s": dial tcp 10.96.141.178:443: connect: connection refused
make: *** [checkpoint-2] Error 1


CalledProcessError: Command 'b'make checkpoint-2\n\n# make checkpoint-2 prepares the cluster for the Kueue topology-aware scheduling exercises.\n\n\n# Phase 1: restore the simulated GPU cluster\n# scripts/checkpoint.sh 2\n# \xe2\x94\x9c\xe2\x94\x80 Creates topology-lab if missing\n# \xe2\x94\x82  \xe2\x94\x94\xe2\x94\x80 cluster-setup/kind-config.yaml\n# \xe2\x94\x9c\xe2\x94\x80 Selects kubectl context kind-topology-lab\n# \xe2\x94\x9c\xe2\x94\x80 Loads cached workshop images\n# \xe2\x94\x9c\xe2\x94\x80 Creates/reconciles namespace training\n# \xe2\x94\x82  \xe2\x94\x94\xe2\x94\x80 cluster-setup/namespaces.yaml\n# \xe2\x94\x9c\xe2\x94\x80 Publishes block, leaf and rack labels\n# \xe2\x94\x82  \xe2\x94\x94\xe2\x94\x80 cluster-setup/label-topology.sh\n# \xe2\x94\x9c\xe2\x94\x80 Advertises one fake GPU on each worker\n# \xe2\x94\x82  \xe2\x94\x94\xe2\x94\x80 cluster-setup/advertise-fake-gpus.sh\n# \xe2\x94\x9c\xe2\x94\x80 Deletes previous trainer and filler Pods\n# \xe2\x94\x82  \xe2\x94\

In [ ]:
%%bash
kubectl get deployment,pods -n kueue-system -o wide

# Manual check for the Kueue controller


In [ ]:
%%bash
kubectl get nodes \
  -L workshop.example.com/gpu-node \
  -L workshop.example.com/leaf

# Check the simulated GPU workers

In [ ]:
%%bash
kubectl get nodes -o go-template='{{range .items}}{{.metadata.name}}{{"  gpu="}}{{index .status.allocatable "workshop.example.com/gpu"}}{{"\n"}}{{end}}'
# Fake GPU Capacity 

## 8 - Read the four-object contract from source

> **Presentation mapping — Live slides 16–20:** Topology is slide 17, ResourceFlavor is slide 18, ClusterQueue/LocalQueue are slide 19, and verification is slide 20.

| Object | Responsibility |
|---|---|
| `Topology` | Orders labels from coarse domain to hostname |
| `ResourceFlavor` | Selects eligible nodes and attaches the topology |
| `ClusterQueue` | Owns quota for the resource/flavor combination |
| `LocalQueue` | Namespace-scoped submission endpoint |

Read the versioned manifests first; then compare them with the live API state.

### Read the four Kueue configuration objects

`LIVE`  `READ-ONLY`

**Why this cell is here**

These four files teach Kueue the location hierarchy, which Nodes may provide our GPU type, how much GPU quota exists, and which queue the `training` namespace uses. The cell displays them without changing the cluster.

**What it does**

- Shows `Topology.spec.levels` as block → leaf → hostname.
- Shows ResourceFlavor node eligibility and topology reference.
- Shows ClusterQueue nominal quota for the fake GPU resource/flavor.
- Shows namespace-scoped LocalQueue pointing to the ClusterQueue.

**Expected result**

Four clearly separated manifests print in the order Topology, ResourceFlavor, ClusterQueue, LocalQueue.

<details>
<summary><strong>Implementation details, interpretation and recovery</strong></summary>

**Runs or reads**

- A Bash loop that prints the four reviewed YAML files in dependency order.

**State changed:** None; these files were applied by Checkpoint 2, but this cell only reads them.

**How to interpret the result:** No single object is 'the topology scheduler.' Their references form the admission contract.

**If it fails:** If an API field is unfamiliar, compare the file with the live object in the next cells. If files are absent, rerun the directory guard.

</details>

In [ ]:
%%bash
for file in kueue-config/topology.yaml kueue-config/resource-flavor.yaml kueue-config/cluster-queue.yaml kueue-config/local-queue.yaml; do printf '
===== %s =====
' "$file"; sed -n '1,220p' "$file"; done


# %%bash
# for file in \
#   kueue-config/topology.yaml \
#   kueue-config/resource-flavor.yaml \
#   kueue-config/cluster-queue.yaml \
#   kueue-config/local-queue.yaml
# do
#   printf '\n===== %s =====\n' "$file"
#   sed -n '1,220p' "$file"
# done


#  Let's you know the config chain
# Topology
# → ResourceFlavor
# → ClusterQueue
# → LocalQueue

### Confirm that Kueue and its queues are ready

`LIVE`  `READ-ONLY`

**Why this cell is here**

Verify controller acceptance and show the cluster-scoped versus namespace-scoped Kueue objects.

**What it does**

- Asserts controller availability, exact Topology level order, ResourceFlavor selector/link, ClusterQueue `Active=True`, and LocalQueue target.
- Lists the live resources after the assertions.

**Expected result**

Ends with `CHECKPOINT 2 VERIFIED`; `gpu-training` is Active and `training/gpu-queue` targets it.

<details>
<summary><strong>Implementation details, interpretation and recovery</strong></summary>

**Runs or reads**

- `scripts/verify-lab2.sh` plus `kubectl get` for the four object kinds.

**State changed:** None.

**How to interpret the result:** Existence is weaker than readiness. The verifier checks semantic links and conditions, not only object names.

**If it fails:** Rerun `make checkpoint-2`. Use the following status-inspection cell to locate an inactive queue or rejected configuration.

</details>

In [ ]:
%%bash
make verify-lab2
kubectl get topology,resourceflavor,clusterqueue
kubectl get localqueue -n training


# kubectl get topology gpu-fabric
# kubectl get resourceflavor h100-topology
# kubectl get clusterqueue gpu-training
# kubectl get localqueue gpu-queue -n training

### Read what the Kueue controllers accepted

`REFERENCE`  `READ-ONLY`

**Why this cell is here**

Learn to inspect reconciled status and reasons instead of assuming an accepted spec.

**What it does**

- Prints controller-populated conditions, admitted/reserved usage, flavors, and reason/message fields while omitting the already-reviewed specs.

**Expected result**

ClusterQueue and LocalQueue conditions report `Active: True`; usage should be zero immediately after the clean checkpoint.

<details>
<summary><strong>Implementation details, interpretation and recovery</strong></summary>

**Runs or reads**

- `kubectl get ... -o yaml` piped through `sed` from the `status:` key onward.

**State changed:** None.

**How to interpret the result:** Spec is requested state; status is the controller's observed result. Reason/message are the first diagnostic surface for inactive queues.

**If it fails:** If `Active=False`, read `reason` and `message`, verify referenced ResourceFlavor/ClusterQueue objects, then rerun `make checkpoint-2`.

</details>

In [ ]:
%%bash
kubectl get clusterqueue gpu-training -o yaml | sed -n '/status:/,$p'
kubectl get localqueue gpu-queue -n training -o yaml | sed -n '/status:/,$p'


---
# Phase 3 - Require, break, and relax locality

> **Presentation mapping — Live slides 21–28:** require one leaf, prove an impossible request waits, allow preferred fallback, and restore a clean TAS-ready state.

This phase changes one policy dimension while retaining the same basic Job shape. The three outcomes are:

1. **Required + feasible:** admit inside one leaf.
2. **Required + impossible:** wait and create zero trainer Pods.
3. **Preferred + impossible at leaf:** widen to block and run across leaves.

## 9 - Required topology: inspect the annotation

> **Presentation mapping — Live slides 22–24:** inspect the required annotation, run the feasible case, then compare Kueue's assignment with the scheduler's Node bindings.

The queue label puts the Job under Kueue. The PodTemplate annotation asks TAS to find one leaf that can fit the complete PodSet.

### Read the strict same-leaf request

`LIVE`  `READ-ONLY`

**Why this cell is here**

This file defines a two-worker training Job that is allowed to start only when both workers fit under the same leaf switch. The command prints just the important lines rather than the entire YAML.

**What it does**

- Prints matching lines with source line numbers: queue label, required-topology annotation, parallelism/completions, and fake-GPU key.

**Expected result**

The output shows queue `gpu-queue`, leaf-level required topology, two replicas, and one fake GPU per Pod.

<details>
<summary><strong>Implementation details, interpretation and recovery</strong></summary>

**Runs or reads**

- `grep -n -E` against `jobs/training-topology-required.yaml`.

**State changed:** None.

**How to interpret the result:** Kueue can calculate a two-GPU PodSet and must find one leaf with two available GPUs before admitting it.

**If it fails:** If expected lines are absent, inspect the whole YAML and run `make test`; do not execute a manifest whose contract is unclear.

</details>

In [ ]:
%%bash
grep -n -E 'queue-name|required-topology|parallelism|completions|workshop.example.com/gpu' jobs/training-topology-required.yaml


### Start a feasible two-trainer same-leaf Job

`LIVE`  `CHANGES LAB`

**Why this cell is here**

Prove that strict leaf locality admits when one leaf can fit the entire two-replica PodSet.

**What it does**

- Clears prior exercises, submits the Job, resolves its generated Workload by ownerReference, waits for `Admitted`, prints the topology assignment, waits for trainer readiness, and asserts one unique leaf.

**Expected result**

Prints `Admitted=True`, a topology assignment, two node→leaf mappings with the same leaf, and `Lab 3 expectation satisfied`.

<details>
<summary><strong>Implementation details, interpretation and recovery</strong></summary>

**Runs or reads**

- `scripts/reset-workloads.sh`.
- `kubectl apply -f jobs/training-topology-required.yaml`.
- `scripts/verify-lab3.sh --expect admitted --job training-topology-aware --same-leaf`.

**State changed:** Creates one Job, a Kueue Workload, and two trainer Pods; reserves two fake GPUs inside one selected leaf.

**How to interpret the result:** Kueue admitted the group into a domain; kube-scheduler then bound individual Pods to eligible Nodes in that domain.

**If it fails:** Run `make checkpoint-2`, then retry. If Pending, inspect Workload conditions and confirm both fake GPUs in at least one leaf are free.

</details>

In [ ]:
%%bash
make lab3-required

# PURPOSE:
# Demonstrate successful topology-aware gang placement.
# Two trainer Pods require one fake GPU each, and both must run in the same
# simulated InfiniBand leaf-switch domain.

# EXECUTION CHAIN:
# Makefile
# └─ target: lab3-required
#    ├─ scripts/reset-workloads.sh
#    │  ├─ Deletes earlier workshop Jobs and their trainer Pods
#    │  ├─ Deletes standalone filler Pods
#    │  └─ Waits until all four fake GPUs are free
#    │
#    ├─ kubectl apply -f jobs/training-topology-required.yaml
#    │  └─ Creates Job/training-topology-aware in namespace training
#    │
#    └─ scripts/verify-lab3.sh
#       ├─ Finds the Kueue Workload owned by the Job
#       ├─ Waits until Workload condition Admitted=True
#       ├─ Verifies QuotaReserved=True
#       ├─ Reads the recorded topologyAssignment
#       ├─ Waits until both trainer Pods are Ready
#       ├─ Finds the Node and leaf label for each Pod
#       └─ Fails unless both Pods are in exactly one leaf

# WHAT THE JOB DECLARES:
# - parallelism: 2
#   Run two trainer Pods at the same time.
#
# - completions: 2
#   The complete distributed workload consists of two replicas.
#
# - completionMode: Indexed
#   Assign stable indexes 0 and 1, similar to distributed-training ranks.
#
# - kueue.x-k8s.io/queue-name: gpu-queue
#   Submit this Job through training/LocalQueue/gpu-queue.
#
# - kueue.x-k8s.io/podset-required-topology:
#     workshop.example.com/leaf
#   Hard requirement: the complete PodSet must fit inside one leaf.
#
# - workshop.example.com/gpu: 1
#   Each trainer requests one simulated GPU.

# WHAT HAPPENS INSIDE KUEUE:
# 1. Kueue sees the Job's gpu-queue label.
# 2. It creates a Workload representing the entire two-Pod Job.
# 3. ClusterQueue/gpu-training checks that two GPU units are available.
# 4. ResourceFlavor/h100-topology identifies eligible GPU workers.
# 5. Topology/gpu-fabric identifies which workers share a leaf.
# 6. Kueue searches for one leaf containing two free fake GPUs.
# 7. It selects either leaf-a or leaf-b.
# 8. It reserves two units of ClusterQueue quota atomically.
# 9. It records the selected domain in Workload.status.topologyAssignment.
# 10. Kueue admits the complete workload and releases the Job.
# 11. kube-scheduler binds both Pods to different workers in the selected leaf.
#
# Because each worker has only one fake GPU, the Pods need two different Nodes.
# However, those two Nodes must have the same leaf label.

# IMPORTANT:
# The containers run `agnhost pause`, not CUDA or NCCL.
# This lab proves Kubernetes/Kueue admission and placement behavior without
# requiring physical GPUs.

In [ ]:
%%bash
kubectl get pods -n training \
  -l job-name=training-topology-aware \
  -o wide

In [ ]:
%%bash
for pod in $(kubectl get pods -n training \
  -l job-name=training-topology-aware \
  -o name)
do
  node=$(kubectl get "$pod" -n training \
    -o jsonpath='{.spec.nodeName}')

  leaf=$(kubectl get node "$node" \
    -o jsonpath='{.metadata.labels.workshop\.example\.com/leaf}')

  printf '%-45s  %-28s  %s\n' \
    "${pod#pod/}" "$node" "$leaf"
done
# Map each Pod to its leaf

In [ ]:
%%bash

# checkpoint/lab reset leaves one active Workload for this exercise.
workload=$(kubectl get workloads -n training \
  -o jsonpath='{.items[0].metadata.name}')

printf '\n=== 1. Requested topology policy ===\n'

kubectl get workload "$workload" -n training \
  -o jsonpath='required-topology={.spec.podSets[0].topologyRequest.required}{"\n"}'

printf '\n=== 2. Topology decision recorded by Kueue ===\n'

kubectl get workload "$workload" -n training \
  -o jsonpath='{range .status.admission.podSetAssignments[*]}Workload='"$workload"' PodSet={.name} levels={.topologyAssignment.levels} slices={.topologyAssignment.slices}{"\n"}{end}'

printf '\n=== 3. Actual Pod bindings ===\n'

kubectl get pods -n training \
  -l job-name=training-topology-aware \
  -o jsonpath='{range .items[*]}{.metadata.name}{" → "}{.spec.nodeName}{"\n"}{end}'

printf '\n=== 4. Verify that the selected Nodes share one leaf ===\n'

for node in $(kubectl get pods -n training \
  -l job-name=training-topology-aware \
  -o jsonpath='{range .items[*]}{.spec.nodeName}{"\n"}{end}')
do
  leaf=$(kubectl get node "$node" \
    -o jsonpath='{.metadata.labels.workshop\.example\.com/leaf}')

  printf '%-28s → %s\n' "$node" "$leaf"
done

printf '\n=== 5. Raw topologyAssignment in the Workload ===\n'

kubectl get workload "$workload" -n training -o yaml |
sed -n '/topologyAssignment:/,/conditions:/p'


# HOW TO READ THE RESULT
#
# required-topology=workshop.example.com/leaf
#   The complete two-Pod training Job must fit inside one leaf domain.
#
# PodSet=main
#   `main` represents the two equivalent trainer Pods belonging to this Job.
#   Kueue evaluates them together instead of admitting each Pod independently.
#
# levels=["kubernetes.io/hostname"]
#   Kueue recorded its final executable decision at individual Node level.
#
# domainCount=2
#   Two hostname domains were selected because the Job needs two Pods and each
#   simulated worker has only one fake GPU.
#
# podCounts={"universal":1}
#   Place one trainer Pod in each selected hostname domain.
#
# prefix="topology-lab-worker", roots=["","2"]
#   This is Kueue's compressed representation of the selected hostnames:
#
#   topology-lab-worker + ""  = topology-lab-worker
#   topology-lab-worker + "2" = topology-lab-worker2
#
# Actual Pod bindings
#   The next output proves that kube-scheduler followed Kueue's assignment.
#
# Node leaf labels
#   Both selected Nodes must report the same leaf, such as leaf-a.
#
# SUCCESS CONDITION
#   Kueue admitted the complete PodSet, reserved two GPU units, selected two
#   Nodes beneath one leaf, and only then allowed both trainer Pods to run.

> Both training Pods were placed on different GPU workers, but those workers belong to the same leaf-switch domain. Kueue evaluated the two replicas as one PodSet, reserved both GPUs together, and selected a valid same-leaf placement before allowing the Job to run. Kubernetes’ scheduler then bound the Pods to those selected Nodes. This is topology-aware scheduling with Kueue.

- Requested policy:
> required-topology = workshop.example.com/leaf

- Kueue decision:
> topology-lab-worker + topology-lab-worker2

- Physical labels:
> both Nodes = leaf-a

In [ ]:
%%bash
kubectl get workloads -n training -o yaml

# Complete status

In [ ]:
%%bash
kubectl describe clusterqueue gpu-training

# Inspect the ClusterQueue Reservation 
# We will see two fake GPU units reserved or used while the trainer Pods are running.


# Output is like:
# Active:               True
# Admitted Workloads:   1
# Pending Workloads:    0
# Nominal Quota:        4
# Flavors Reservation:  2
# Flavors Usage:        2
# Borrowed:             0


# That means:
# - gpu-training is healthy and accepting work.
# - One distributed training Workload was admitted.
# - The Workload reserved two fake GPUs together.
# - Both fake GPUs are currently being used.
# - Two of the queue’s four GPU units remain free.
# - Nothing was borrowed from another queue.
# - No Workload is waiting.

### Prove Kueue's assignment and the final Pod placement agree

`LIVE`  `READ-ONLY`

**Why this cell is here**

Correlate Kueue's admitted topology intent with kube-scheduler's actual Pod bindings.

**What it does**

- Shows Workload admission state.
- Extracts persisted assignment domains/counts.
- Shows Pod node names and maps those Nodes to leaves.

**Expected result**

The Workload is admitted; assigned domain/count cover two replicas; both Pods are Ready on one leaf.

<details>
<summary><strong>Implementation details, interpretation and recovery</strong></summary>

**Runs or reads**

- `kubectl` reads for Workload summaries, `topologyAssignment`, trainer Pods, and Node leaf labels.

**State changed:** None.

**How to interpret the result:** Admission intent and actual placement agree. If only one layer were checked, controller or scheduler drift could remain invisible.

**If it fails:** Use `kubectl describe workload -n training <name>` and `kubectl describe pod -n training <name>`, then restore with `make checkpoint-2`.

</details>

In [ ]:
%%bash
kubectl get workloads -n training
kubectl get workloads -n training -o yaml | sed -n '/topologyAssignment/,+18p'
kubectl get pods -n training -l job-name=training-topology-aware -o wide
kubectl get nodes -L workshop.example.com/leaf


**Expected evidence:**

- Workload condition `Admitted=True`;
- a persisted `topologyAssignment` at leaf level;
- both trainer Pods bound to nodes carrying the same leaf label.

This is domain-aware admission followed by ordinary scheduler binding. Kueue is not a second kube-scheduler.

## Lab 3 recap : from topology data to topology-aware placement

### 1. The simulated GPU datacenter

Checkpoint 1 created five Kubernetes Nodes:

- One control-plane Node.
- Four worker Nodes.
- One simulated GPU on every worker.
- Two workers under each simulated leaf switch.

```text
                         block-1
               ┌────────────┴────────────┐
             leaf-a                   leaf-b
       ┌────────┴────────┐       ┌────────┴────────┐
     worker           worker2   worker3          worker4
      1 GPU             1 GPU    1 GPU             1 GPU
```

The topology comes from Node labels:

```text
worker   → block-1 → leaf-a
worker2  → block-1 → leaf-a
worker3  → block-1 → leaf-b
worker4  → block-1 → leaf-b
```

At this stage, Kubernetes could see four GPU resources, but the default scheduler did not understand that the GPUs belonged to different leaf-switch domains.

---

### 2. Checkpoint 2 taught Kueue about this topology

`make checkpoint-2` installed Kueue and created four connected objects:

```text
Topology/gpu-fabric
        │
        ▼
ResourceFlavor/h100-topology
        │
        ▼
ClusterQueue/gpu-training
        │
        ▼
training/LocalQueue/gpu-queue
```

#### `Topology/gpu-fabric`

Defines the hierarchy Kueue should use:

```text
block → leaf → hostname
```

The Topology object contains label keys, not the actual names `leaf-a`, `leaf-b`, or `worker2`.

Kueue reads those values from the Kubernetes Node labels.

#### `ResourceFlavor/h100-topology`

Selects Nodes carrying:

```yaml
workshop.example.com/gpu-node: "true"
```

It connects those eligible GPU workers to `Topology/gpu-fabric`.

#### `ClusterQueue/gpu-training`

Provides admission quota for four simulated GPUs:

```yaml
resource: workshop.example.com/gpu
flavor: h100-topology
nominalQuota: 4
```

#### `LocalQueue/gpu-queue`

Provides a queue inside the `training` namespace.

A Job selects it using:

```yaml
kueue.x-k8s.io/queue-name: gpu-queue
```

At the end of Checkpoint 2:

```text
✓ Kueue controller running
✓ Four GPU workers eligible
✓ Four fake GPUs available
✓ Topology hierarchy registered
✓ ClusterQueue Active=True
✓ No training workload running
```

---

### 3. Lab 3 declared a topology requirement

`make lab3-required` applied:

```text
jobs/training-topology-required.yaml
```

The Job declared:

```yaml
parallelism: 2
completions: 2
```

This represents two trainer replicas.

Each trainer requested one fake GPU:

```yaml
resources:
  requests:
    workshop.example.com/gpu: 1
  limits:
    workshop.example.com/gpu: 1
```

The queue label submitted the Job through Kueue:

```yaml
kueue.x-k8s.io/queue-name: gpu-queue
```

The critical annotation required the entire PodSet to fit inside one leaf:

```yaml
kueue.x-k8s.io/podset-required-topology:
  workshop.example.com/leaf
```

The resulting requirement was:

```text
2 trainer Pods
× 1 GPU each
= 2 GPUs required inside one leaf
```

---

### 4. What happened under the hood

```text
Job/training-topology-aware
        │
        │ queue-name=gpu-queue
        ▼
LocalQueue/gpu-queue
        │
        ▼
ClusterQueue/gpu-training
        │
        │ Check aggregate quota
        │ 2 requested ≤ 4 available
        ▼
ResourceFlavor/h100-topology
        │
        │ Find eligible GPU workers
        ▼
Topology/gpu-fabric
        │
        │ Search for one leaf with 2 free GPUs
        ▼
Reserve one valid topology assignment
        │
        ▼
Admit the complete Workload
        │
        ▼
kube-scheduler binds the two Pods
```

Kueue evaluated the two trainers as one PodSet. It did not admit them independently.

Both `leaf-a` and `leaf-b` originally had two available GPUs, so either leaf was a valid result.

---

### 5. The topology decision on this cluster

Kueue selected:

```text
topology-lab-worker
topology-lab-worker2
```

The resulting placement was:

```text
                         block-1
               ┌────────────┴────────────┐
             leaf-a                   leaf-b
       ┌────────┴────────┐       ┌────────┴────────┐
     worker           worker2   worker3          worker4
   trainer-0         trainer-1    free              free
     1 GPU             1 GPU     1 GPU             1 GPU
```

The two trainers run on different Nodes because each worker has only one fake GPU.

However, both selected Nodes belong to `leaf-a`, which satisfies the hard same-leaf requirement.

Another laptop could validly select `worker3` and `worker4` under `leaf-b`.

---

### 6. How to decode Kueue’s assignment

The Workload reported:

```text
levels=["kubernetes.io/hostname"]

domainCount=2

podCounts={"universal":1}

prefix="topology-lab-worker"

roots=["","2"]
```

Kueue compresses repeated hostname prefixes:

```text
topology-lab-worker + ""  = topology-lab-worker
topology-lab-worker + "2" = topology-lab-worker2
```

Therefore, the assignment means:

```text
Two hostname domains were selected.
Place one trainer Pod on each selected hostname.
```

The original same-leaf requirement is stored separately in:

```text
spec.podSets[].topologyRequest.required
```

with the value:

```text
workshop.example.com/leaf
```

The assignment contains the concrete hostnames. Their Node labels prove that both hostnames are under `leaf-a`.

---

### 7. What the ClusterQueue status means

The ClusterQueue reported:

```text
Active=True
Admitted Workloads=1
Pending Workloads=0
Nominal Quota=4
Flavors Reservation Total=2
Flavors Usage Total=2
Borrowed=0
```

This means:

- The queue is healthy.
- One Workload was admitted.
- The complete Workload reserved two fake GPUs.
- Both reserved GPUs are currently in use.
- Two of the queue's four GPU units remain available.
- No quota was borrowed from another queue.
- No Workload is waiting.

---

### 8. Control-plane decision versus actual execution

We verify the result at two layers:

```text
CONTROL-PLANE EVIDENCE

Kueue Workload:
✓ QuotaReserved=True
✓ Admitted=True
✓ Resource usage=2 GPUs
✓ Flavor=h100-topology
✓ Assigned worker and worker2
```

```text
EXECUTION EVIDENCE

Kubernetes Pods:
✓ trainer-0 running on worker
✓ trainer-1 running on worker2
✓ worker and worker2 both labeled leaf-a
```

Kueue and kube-scheduler have different responsibilities:

```text
Kueue
├── Evaluates the complete PodSet
├── Checks quota
├── Checks topology capacity
├── Reserves an allowed assignment
└── Admits the Workload

kube-scheduler
├── Receives the admitted Pods
├── Applies Kueue's placement constraints
└── Performs the final Pod-to-Node binding
```

Kueue does not replace kube-scheduler.

---

### 9. What Lab 3A proved

```text
Default Kubernetes scheduling:
Two independent GPU requests may cross leaf boundaries.

Kueue Topology-Aware Scheduling:
The complete distributed Job is evaluated before its Pods start.
```

Lab 3A proved that:

> Kueue can reserve capacity for a complete two-replica training Job and keep both replicas inside one required leaf-switch domain.

This lab proves control-plane admission and placement behavior. It does not measure real NVLink, InfiniBand, NCCL, or GPU performance because the workshop uses simulated GPU resources and `agnhost pause` containers.

---

### 10. The remaining Lab 3 experiments

The complete Lab 3 compares three policies:

```text
A. REQUIRED + FEASIBLE

2 replicas need 2 GPUs in one leaf
→ Workload admitted
→ 2 Pods created
→ Both Pods use the same leaf
```

```text
B. REQUIRED + IMPOSSIBLE

3 replicas need 3 GPUs in one leaf
→ Neither leaf has enough capacity
→ Workload waits
→ 0 trainer Pods created
```

```text
C. PREFERRED + IMPOSSIBLE AT LEAF

3 replicas prefer one leaf
→ No leaf has enough capacity
→ Kueue widens to block-1
→ Workload admitted
→ Pods span leaf-a and leaf-b
```

The policy lesson is:

> `required` preserves the topology boundary even if the Job must wait.  
> `preferred` tries locality first but may widen when starting the Job is more important.

## 10 - Required topology: deliberately make it impossible

> **Presentation mapping - Live slide 25:** run this case immediately after explaining that neither two-GPU leaf can hold three replicas.

Each leaf contains only two fake GPUs. This Job requests three replicas and requires one leaf. The correct outcome is waiting—not silently violating locality and not starting a partial training group.

### Submit a three-trainer Job that cannot fit in one leaf

`LIVE`  `CHANGES LAB`

**Why this cell is here**

The YAML asks for three fake GPUs in one leaf, but each leaf has only two. The first command shows that request; the second submits it and proves that no trainer Pod starts.

**What it does**

- Confirms three one-GPU replicas require one leaf although each leaf owns only two GPUs.
- Submits the Job, waits for an explicit negative admission/quota condition, and asserts zero trainer Pods exist.

**Expected result**

Prints `required topology held the impossible workload before GPU allocation`; no trainer Pod is created.

<details>
<summary><strong>Implementation details, interpretation and recovery</strong></summary>

**Runs or reads**

- A focused manifest inspection, then `scripts/reset-workloads.sh`, `kubectl apply`, and `verify-lab3.sh --expect pending`.

**State changed:** Replaces the prior exercise with a suspended Job and pending Kueue Workload. It should allocate zero fake GPUs.

**How to interpret the result:** Pending is the correct preservation of a required service contract, not necessarily a malfunction.

**If it fails:** If trainer Pods exist, stop and run `make checkpoint-2`; if the verifier times out, inspect Workload conditions and Kueue controller logs.

</details>

In [ ]:
%%bash
grep -n -E 'required-topology|parallelism|completions|workshop.example.com/gpu' jobs/training-impossible.yaml
make lab3-impossible


# PURPOSE:
# Demonstrate fail-closed topology-aware admission.
#
# This Job requests three trainer replicas inside one required leaf, but each
# leaf contains only two simulated GPUs. Kueue must keep the complete Workload
# waiting and create zero trainer Pods.


# WHAT THE GREP COMMAND DOES:
#
# grep
#   Searches the manifest without changing it.
#
# -n
#   Prints the source line number.
#
# -E
#   Enables the `|` operator, meaning "match any of these fields".
#
# The command displays only the fields relevant to this experiment:
#
# parallelism: 3
#   The Job wants three trainer Pods running simultaneously.
#
# completions: 3
#   The complete distributed Job contains three trainer replicas.
#
# podset-required-topology: workshop.example.com/leaf
#   All three replicas must fit inside one leaf-switch domain.
#
# workshop.example.com/gpu: 1
#   Every trainer requests and limits one fake GPU.


# CAPACITY VERSUS DEMAND:
#
#                         block-1: 4 GPUs
#               ┌────────────┴────────────┐
#             leaf-a                   leaf-b
#              2 GPUs                    2 GPUs
#
# Job requirement:
#
#   3 replicas × 1 GPU = 3 GPUs inside one leaf
#
# Aggregate cluster capacity is sufficient:
#
#   3 requested <= 4 total GPUs
#
# But topology capacity is insufficient:
#
#   3 requested > 2 GPUs in leaf-a
#   3 requested > 2 GPUs in leaf-b
#
# This is a topology failure, not an aggregate quota shortage.


# EXECUTION CHAIN:
#
# Makefile
# └─ target: lab3-impossible
#    │
#    ├─ scripts/reset-workloads.sh
#    │  ├─ Deletes the previous topology-aware Job
#    │  ├─ Kubernetes deletes its owned trainer Pods and Workload
#    │  ├─ Deletes any filler Pods
#    │  └─ Waits until all four fake GPUs are free
#    │
#    ├─ kubectl apply -f jobs/training-impossible.yaml
#    │  └─ Creates Job/training-impossible
#    │
#    └─ scripts/verify-lab3.sh
#       ├─ Finds the Workload owned by Job/training-impossible
#       ├─ Waits for an explicit pending/unreserved condition
#       ├─ Confirms that quota was not reserved
#       ├─ Confirms that the Workload was not admitted
#       └─ Confirms that zero trainer Pods were created


# WHAT KUEUE DOES:
#
# 1. The Job enters through LocalQueue/gpu-queue.
#
# 2. Kueue creates one Workload containing a PodSet of three trainers.
#
# 3. ClusterQueue/gpu-training has enough aggregate quota for three GPUs.
#
# 4. Kueue evaluates the required leaf-level topology.
#
# 5. leaf-a has only two GPU units.
#
# 6. leaf-b also has only two GPU units.
#
# 7. No leaf can satisfy the complete three-GPU PodSet.
#
# 8. Kueue does not split the hard-constrained PodSet across both leaves.
#
# 9. Kueue does not reserve quota or admit only two of the three trainers.
#
# 10. The Workload remains waiting and Kubernetes creates zero trainer Pods.


# HOW TO READ THE OUTPUT:
#
# job.batch "training-topology-aware" deleted
#   The previous Lab 3A Job was removed so it cannot consume GPU capacity.
#
# No resources found
#   No filler Pods existed to delete. This is normal, not an error.
#
# OK all four fake GPUs are available
#   The experiment started with a clean four-GPU baseline.
#
# job.batch/training-impossible created
#   Kubernetes accepted the Job object. This does not mean the Job was admitted
#   or that trainer Pods were allowed to start.
#
# Admitted=Unknown
#   The Workload does not currently have a positive Admitted condition.
#   It must not be interpreted as admitted.
#
# QuotaReserved=False
#   Kueue explicitly refused to reserve GPU quota because no valid same-leaf
#   topology assignment exists.
#
# OK required topology held the impossible workload before GPU allocation
#   This is the expected successful result of this experiment.


# WHY ZERO PODS IS IMPORTANT:
#
# A three-replica synchronous training Job cannot make progress with only two
# trainers. Starting two Pods would consume GPUs without producing useful work.
#
# Kueue therefore evaluates the whole PodSet before any trainer starts:
#
#   Incorrect partial outcome:
#     2 Pods Running + 1 Pod Pending
#
#   Correct Kueue outcome:
#     0 Pods Running + complete Workload waiting


# SUCCESS CONDITION:
#
# Workload exists       = yes
# Aggregate quota       = sufficient
# Required leaf capacity = insufficient
# QuotaReserved         = False
# Trainer Pods created  = 0
#
# The pending Workload is not a failure. It proves that Kueue preserved the
# hard topology promise instead of silently degrading placement.

### Prove that strict locality created zero trainer Pods

`LIVE`  `READ-ONLY`

**Why this cell is here**

Distinguish policy-preserving Pending from a scheduler failure.

**What it does**

- Prints admission/quota condition reason and message.
- Queries trainer Pods using the Job label.

**Expected result**

The Workload is not admitted/reserved and the final Pod query returns no rows.

<details>
<summary><strong>Implementation details, interpretation and recovery</strong></summary>

**Runs or reads**

- Workload summary and condition reads plus a Pod query for the impossible Job.

**State changed:** None.

**How to interpret the result:** Kueue evaluated the complete PodSet and withheld admission; kube-scheduler never received partial trainer Pods to bind.

**If it fails:** If no Workload exists, verify the queue label and controller. If Pods exist, restore Checkpoint 2 before continuing.

</details>

In [161]:
%%bash
kubectl get workloads -n training
kubectl get workloads -n training -o yaml | sed -n '/conditions:/,+22p'
kubectl get pods -n training -l job-name=training-impossible


No resources found in training namespace.
No resources found in training namespace.


## 11 — Preferred topology: widen the admissible domain

> **Presentation mapping — Live slides 26–28:** explain the business promise, show the one-word manifest change, run preferred fallback, and finish with Checkpoint 3.

The preferred case requests the same three replicas. Only the annotation key changes. TAS attempts leaf locality first and may widen to the enclosing block.

### Compare required locality with preferred locality

`LIVE`  `READ-ONLY`

**Why this cell is here**

This compares the waiting Job with the fallback Job so you can see that the important change is `required` becoming `preferred`.

**What it does**

- Compares the strict and preferred Job manifests line by line.

**Expected result**

The meaningful change is the topology annotation from `required` to `preferred` (names/labels may also differ to keep experiments distinct).

<details>
<summary><strong>Implementation details, interpretation and recovery</strong></summary>

**Runs or reads**

- Unified `diff`; `|| true` keeps Jupyter from treating expected differences as a failed cell.

**State changed:** None.

**How to interpret the result:** The resource demand remains impossible at leaf scope; only permission to widen changes.

**If it fails:** If resource counts differ, inspect both complete files before presenting—the comparison would no longer isolate policy.

</details>

In [162]:
%%bash
diff -u jobs/training-impossible.yaml jobs/training-topology-preferred.yaml || true


--- jobs/training-impossible.yaml	2026-09-09 17:50:33
+++ jobs/training-topology-preferred.yaml	2026-09-09 17:50:33
@@ -1,26 +1,28 @@
-# PURPOSE: Prove that infeasible *required* locality waits before allocating GPUs.
-# USED BY: `make lab3-impossible`; compare with training-topology-preferred.yaml.
-# PRECONDITION: Checkpoint 2 is healthy; each leaf owns only 2 fake GPUs.
-# CONTRACT: Three one-GPU replicas must all fit under one leaf value.
-# EXPECTED RESULT: No leaf can supply 3 GPUs, so the Workload remains
-# unadmitted/unreserved and Kubernetes creates zero trainer Pods.
-# WHY THIS IS SUCCESS: Pending preserves the submitted hard requirement. Starting
-# two of three synchronous ranks or silently crossing leaves would violate it.
+# PURPOSE: Prove that *preferred* locality may widen to start an otherwise
+# infeasible Job; compare directly with training-impossible.yaml.
+# USED BY: `make lab3-preferred`, which resets earlier workshop workloads first.
+# PRECONDITION: Checkpoint

### Allow the three-trainer Job to widen across both leaves

`LIVE`  `CHANGES LAB`

**Why this cell is here**

Show preferred locality widening from an infeasible two-GPU leaf to the four-GPU enclosing block.

**What it does**

- Clears prior workloads, submits the three-replica preferred Job, waits for admission/readiness, and asserts trainers span at least two leaves.
- Prints the persisted topology assignment and actual bindings.

**Expected result**

The Workload is admitted, the assignment widens, and Ready Pods appear across at least two leaf labels.

<details>
<summary><strong>Implementation details, interpretation and recovery</strong></summary>

**Runs or reads**

- `make lab3-preferred` followed by Workload, Pod, and Node evidence queries.

**State changed:** Creates a Job, Workload, and three trainer Pods consuming three of four fake GPUs across the block.

**How to interpret the result:** Preferred topology is a business trade-off: start with a wider domain when strict locality cannot be satisfied.

**If it fails:** Run `make checkpoint-2` and retry. If the Job waits unexpectedly, inspect free allocatable capacity and Workload condition messages.

</details>

In [163]:
%%bash
make lab3-preferred
kubectl get workloads -n training -o yaml | sed -n '/topologyAssignment/,+18p'
kubectl get pods -n training -l job-name=training-preferred -o wide
kubectl get nodes -L workshop.example.com/leaf


bash scripts/reset-workloads.sh
==> Deleting workshop Jobs and deterministic filler Pods
No resources found
No resources found
OK  all four fake GPUs are available for the next exercise
kubectl apply -f jobs/training-topology-preferred.yaml
job.batch/training-preferred created
bash scripts/verify-lab3.sh --expect admitted --job training-preferred --cross-leaf


error: timed out waiting for the condition on workloads/job-training-preferred-a1f2e
make: *** [lab3-preferred] Error 1
No resources found in training namespace.


NAME                         STATUS   ROLES           AGE   VERSION   LEAF
topology-lab-control-plane   Ready    control-plane   13m   v1.34.0   
topology-lab-worker          Ready    <none>          13m   v1.34.0   leaf-a
topology-lab-worker2         Ready    <none>          13m   v1.34.0   leaf-a
topology-lab-worker3         Ready    <none>          13m   v1.34.0   leaf-b
topology-lab-worker4         Ready    <none>          13m   v1.34.0   leaf-b


### Required versus preferred is a business contract

```text
completion time = queue delay + execution time
```

Suppose one leaf can fit the job in three minutes, while a cross-leaf placement can start now but trains 35% more slowly. The right choice depends on deadline, expected duration, fragmentation, priority, and cost—not on a universal “always pack” rule.

Use **required** when violating the boundary is unacceptable. Use **preferred** when the cost of waiting may exceed the performance penalty.

### Remove the exercises but keep Kueue ready

`LIVE`  `CHANGES LAB`

**Why this cell is here**

Keep Kueue/TAS installed while removing every exercise workload before the production-reference sections.

**What it does**

- Revalidates/reconciles cluster topology, fake GPU capacity, Kueue installation, eligibility labels, and all four Kueue objects.
- Deletes lab Jobs/Pods and waits for resource release.

**Expected result**

Outputs all three checkpoint banners, ending with `CHECKPOINT 3 — clean TAS-ready state restored`.

<details>
<summary><strong>Implementation details, interpretation and recovery</strong></summary>

**Runs or reads**

- `scripts/checkpoint.sh 3`, which deliberately composes Checkpoints 1 and 2 before its final reset.

**State changed:** May reapply known configuration and deletes only workshop workloads; the clean Kueue/TAS control plane remains.

**How to interpret the result:** This is the strongest recovery boundary: known infrastructure, known admission configuration, zero lab workload demand.

**If it fails:** Rerun it. If it fails before Checkpoint 2, diagnose the earlier named checkpoint rather than continuing.

</details>

In [164]:
%%bash
make checkpoint-3


# PURPOSE:
# Restore a clean, verified, topology-aware cluster after all Lab 3 experiments.
#
# Lab 3 created several long-running or waiting Jobs:
#
# - training-topology-aware
# - training-impossible
# - training-preferred
#
# The running Jobs use `agnhost pause`, so they do not finish naturally.
# Checkpoint 3 removes those workloads while preserving the cluster, fake GPU
# topology, Kueue installation and queue configuration.


# EXECUTION CHAIN:
#
# Makefile
# └─ target: checkpoint-3
#    └─ scripts/checkpoint.sh 3
#       │
#       ├─ PHASE 1: restore the simulated GPU datacenter
#       │  ├─ Creates topology-lab if it is missing
#       │  │  └─ cluster-setup/kind-config.yaml
#       │  ├─ Selects kubectl context kind-topology-lab
#       │  ├─ Loads the cached workshop images
#       │  │  └─ scripts/load-images.sh
#       │  ├─ Creates or reconciles namespace training
#       │  │  └─ cluster-setup/namespaces.yaml
#       │  ├─ Validates and publishes block/leaf/rack labels
#       │  │  └─ cluster-setup/label-topology.sh
#       │  ├─ Advertises one fake GPU on every worker
#       │  │  └─ cluster-setup/advertise-fake-gpus.sh
#       │  ├─ Deletes existing workshop Jobs and Pods
#       │  │  └─ scripts/reset-workloads.sh
#       │  └─ Verifies the five-node fake GPU topology
#       │     └─ scripts/verify-lab1.sh
#       │
#       ├─ PHASE 2: restore Kueue and Topology-Aware Scheduling
#       │  ├─ Installs/reconciles Kueue from the local cache
#       │  │  └─ scripts/install-kueue.sh
#       │  ├─ Labels all four workers as eligible GPU Nodes
#       │  │  └─ cluster-setup/label-gpu-nodes.sh
#       │  ├─ Applies Topology/gpu-fabric
#       │  │  └─ kueue-config/topology.yaml
#       │  ├─ Applies ResourceFlavor/h100-topology
#       │  │  └─ kueue-config/resource-flavor.yaml
#       │  ├─ Applies ClusterQueue/gpu-training
#       │  │  └─ kueue-config/cluster-queue.yaml
#       │  ├─ Applies training/LocalQueue/gpu-queue
#       │  │  └─ kueue-config/local-queue.yaml
#       │  └─ Verifies the complete Kueue object chain
#       │     └─ scripts/verify-lab2.sh
#       │
#       └─ PHASE 3: perform a final workload cleanup
#          └─ scripts/reset-workloads.sh
#             ├─ Deletes all Jobs labeled workshop.example.com/lab
#             ├─ Kubernetes garbage-collects their owned Pods and Workloads
#             ├─ Deletes standalone filler Pods
#             └─ Waits until the associated Pods are gone


# WHY CHECKPOINT 3 REPEATS CHECKPOINTS 1 AND 2:
#
# It is a recovery checkpoint, not merely a delete command.
#
# A participant may reach this point with:
#
# - missing Node labels;
# - fake GPU capacity that was not republished;
# - a missing or unhealthy Kueue controller;
# - an inactive ClusterQueue;
# - stale Jobs or Pods from an earlier exercise.
#
# Checkpoint 3 repairs the complete environment before performing final cleanup.
# Reapplying declarative Kubernetes objects updates existing objects rather than
# creating duplicate queues or topology definitions.


# WHAT CHECKPOINT 3 DELETES:
#
# - Lab 1 topology-unaware Job
# - Lab 3 required-topology Job
# - Lab 3 impossible Job
# - Lab 3 preferred-topology Job
# - Trainer Pods owned by those Jobs
# - Standalone placement-filler Pods
# - Kueue Workloads owned by deleted Jobs, through garbage collection


# WHAT CHECKPOINT 3 PRESERVES:
#
# - The topology-lab kind cluster
# - The control-plane and four worker Nodes
# - Node block, leaf and rack labels
# - One fake GPU advertised by every worker
# - The Kueue controller
# - Topology/gpu-fabric
# - ResourceFlavor/h100-topology
# - ClusterQueue/gpu-training
# - training/LocalQueue/gpu-queue
# - Cached container images
# - The cached Kueue installation manifest


# EXPECTED FINAL TOPOLOGY:
#
#                         block-1
#               ┌────────────┴────────────┐
#             leaf-a                   leaf-b
#       ┌────────┴────────┐       ┌────────┴────────┐
#     worker           worker2   worker3          worker4
#    1 GPU free       1 GPU free 1 GPU free       1 GPU free
#
# No trainer or filler Pods should remain.


# EXPECTED SUCCESS OUTPUT:
#
# CHECKPOINT 1 — fake GPU datacenter ready
#   The Nodes, labels and fake GPU capacity are healthy.
#
# CHECKPOINT 2 — Kueue + TAS ready
#   The controller, Topology, ResourceFlavor and queues are healthy.
#
# CHECKPOINT 3 — clean TAS-ready state restored
#   Lab workloads are removed, but the environment remains ready for another
#   topology-aware experiment.


# SUCCESS CONDITION:
#
# Cluster                       = running
# Worker Nodes                  = 4 Ready
# Fake GPU capacity             = 4 available
# Kueue controller              = Available
# ClusterQueue/gpu-training     = Active=True
# LocalQueue/gpu-queue          = Active
# Trainer Pods                  = 0
# Filler Pods                   = 0
# Active workshop GPU usage     = 0
#
# Checkpoint 3 is a clean recovery boundary, not a cluster teardown.


bash scripts/checkpoint.sh 3
==> Loading registry.k8s.io/e2e-test-images/agnhost:2.53 into topology-lab


Image: "registry.k8s.io/e2e-test-images/agnhost:2.53" with ID "sha256:6debb3645a281a2a03eb859d3fb46a9baf27218aaf817d3e95505db1ec170f8d" found to be already present on all nodes.


==> Loading registry.k8s.io/kueue/kueue:v0.19.2 into topology-lab


Image: "registry.k8s.io/kueue/kueue:v0.19.2" with ID "sha256:656064ccfc30ea3a0eed8ef34a50a96652c89c30cc33ce93f7639c7be3458ba9" found to be already present on all nodes.


namespace/training unchanged
OK: 4 nodes satisfy topology inventory v1
==> Labeling topology-lab-worker -> block=block-1 leaf=leaf-a rack=rack-a1
node/topology-lab-worker not labeled
==> Labeling topology-lab-worker2 -> block=block-1 leaf=leaf-a rack=rack-a1
node/topology-lab-worker2 not labeled
==> Labeling topology-lab-worker3 -> block=block-1 leaf=leaf-b rack=rack-b1
node/topology-lab-worker3 not labeled
==> Labeling topology-lab-worker4 -> block=block-1 leaf=leaf-b rack=rack-b1
node/topology-lab-worker4 not labeled
NAME                         STATUS   ROLES           AGE   VERSION   BLOCK     LEAF     RACK
topology-lab-control-plane   Ready    control-plane   13m   v1.34.0                      
topology-lab-worker          Ready    <none>          13m   v1.34.0   block-1   leaf-a   rack-a1
topology-lab-worker2         Ready    <none>          13m   v1.34.0   block-1   leaf-a   rack-a1
topology-lab-worker3         Ready    <none>          13m   v1.34.0   block-1   leaf-b   rack-b1


In [165]:
%%bash

printf '\n=== Nodes and topology ===\n'

kubectl get nodes \
  -L workshop.example.com/block \
  -L workshop.example.com/leaf \
  -L workshop.example.com/gpu-node

printf '\n=== Kueue controller ===\n'

kubectl get deployment,pods -n kueue-system

printf '\n=== Kueue configuration remains installed ===\n'

kubectl get topologies,resourceflavors,clusterqueues
kubectl get localqueues -n training

printf '\n=== Workshop Jobs: expected none ===\n'

kubectl get jobs -n training \
  -l workshop.example.com/lab

printf '\n=== Workshop Pods: expected none ===\n'

kubectl get pods -n training \
  -l workshop.example.com/lab

printf '\n=== Kueue Workloads: expected none after garbage collection ===\n'

kubectl get workloads -n training

printf '\n=== Final automated verification ===\n'

make verify-lab2


=== Nodes and topology ===
NAME                         STATUS   ROLES           AGE   VERSION   BLOCK     LEAF     GPU-NODE
topology-lab-control-plane   Ready    control-plane   13m   v1.34.0                      
topology-lab-worker          Ready    <none>          13m   v1.34.0   block-1   leaf-a   true
topology-lab-worker2         Ready    <none>          13m   v1.34.0   block-1   leaf-a   true
topology-lab-worker3         Ready    <none>          13m   v1.34.0   block-1   leaf-b   true
topology-lab-worker4         Ready    <none>          13m   v1.34.0   block-1   leaf-b   true

=== Kueue controller ===
NAME                                       READY   UP-TO-DATE   AVAILABLE   AGE
deployment.apps/kueue-controller-manager   1/1     1            1           2m50s

NAME                                            READY   STATUS    RESTARTS   AGE
pod/kueue-controller-manager-79844ddf98-42hnc   1/1     Running   0          2m50s

=== Kueue configuration remains installed ===
NAME    

No resources found in training namespace.



=== Workshop Pods: expected none ===


No resources found in training namespace.



=== Kueue Workloads: expected none after garbage collection ===


No resources found in training namespace.



=== Final automated verification ===
bash scripts/verify-lab2.sh
OK  Kueue controller available
OK  Topology levels: workshop.example.com/block workshop.example.com/leaf kubernetes.io/hostname
OK  ResourceFlavor selects GPU nodes and references gpu-fabric
OK  ClusterQueue gpu-training Active=True
OK  training/gpu-queue -> gpu-training
OK  CHECKPOINT 2 VERIFIED — Kueue topology-aware admission ready


> Checkpoint 3 does not destroy the lab. It reconstructs and verifies the fake GPU topology, revalidates the Kueue configuration, and then removes every experimental workload. We finish with four free GPUs, an active topology-aware queue, and zero trainer Pods. This gives us a known recovery point from which any experiment can be repeated.

---
# Phase 4 — Do not collapse four different topology mechanisms

> **Presentation mapping — Live slides 29–31:** this is a guided explanation section. No additional live workload is required.

| Mechanism | Question it answers | Scope |
|---|---|---|
| Kueue TAS | Which domain can fit the whole PodSet? | Cluster admission |
| Gang scheduling | Can the required members start as a group? | Workload start semantics |
| Kueue Cohort | Which ClusterQueues may borrow quota? | Queue economics |
| Topology Manager | Can CPU, memory and devices align on this node? | kubelet / NUMA |

All-or-nothing start and locality often matter together, but they are not synonyms. A Cohort is quota sharing—not gang scheduling.

## 12 — Why partial placement is dangerous

> **Presentation mapping — Live slides 29–30:** use the partial-start example to separate gang semantics, Kueue Workload admission, TAS, and Cohort quota sharing.

Synchronous training may show several Running Pods while doing no useful training because the missing ranks prevent rendezvous or collective progress. That is why the evidence must include:

- required membership and admitted PodSet size;
- topology assignment;
- actual Pod node/leaf labels;
- application progress, not only Pod phase.

Elastic training is different: starting at a smaller membership is safe only if the application explicitly supports the corresponding batch, optimizer, checkpoint, and rendezvous semantics.

## 13 — Node-local NUMA alignment

> **Presentation mapping — Live slide 31:** this fragment is illustrative and is not applied to the kind cluster.

After cluster placement, kubelet Topology Manager coordinates device, CPU, and memory hints on each node. Common policies include `none`, `best-effort`, `restricted`, and `single-numa-node`.

```yaml
# Illustrative kubelet fragment—not applied to the kind lab.
topologyManagerPolicy: single-numa-node
topologyManagerScope: pod
cpuManagerPolicy: static
```

TAS cannot guarantee NUMA-local GPU/NIC placement. Topology Manager cannot choose a rack or leaf. A production platform needs both layers.

---
# Phase 5 — Treat topology as production data

> **Presentation mapping — Live slides 32–33:** connect authoritative ownership to the sync controller's reconciliation and drift behavior.

The label pipeline is a reconciliation system, not a one-time script:

```text
LLDP + fabric manager + DCIM + GPU/device inventory
                         ↓ normalize
              versioned topology contract
                         ↓ validate
              labels / device metadata / health
                         ↓ reconcile
             Kueue, policy, metrics, operations
```

Every published fact needs an owner, provenance, freshness, conflict policy, and controlled mutation path.

## 14 — Schema failure exercise

> **Presentation mapping — Live slides 32–33:** this optional exercise demonstrates why topology data is validated before publication.

This cell mutates an in-memory copy of the sample inventory. It does not change the repository. The expected error proves that malformed topology is rejected before labels reach Kubernetes.

### Break one inventory value and watch validation reject it

`OPTIONAL`

**Why this cell is here**

Experience a failed data-contract validation without editing the source inventory or touching Kubernetes.

**What it does**

- Loads valid source data, deep-copies it, replaces one leaf value with spaces/uppercase characters, and validates the broken copy against Draft 2020-12 JSON Schema.
- Treats rejection as the expected outcome.

**Expected result**

Prints `EXPECTED REJECTION:` followed by the label-pattern validation message.

<details>
<summary><strong>Implementation details, interpretation and recovery</strong></summary>

**Runs or reads**

- Python standard library plus the installed `jsonschema` package.
- The checked-in inventory and JSON Schema files.

**State changed:** Only a Python object in kernel memory. No file or Kubernetes object is modified.

**How to interpret the result:** Malformed physical facts are rejected before publication. This schema check complements the renderer's semantic checks such as duplicates and provenance.

**If it fails:** If `jsonschema` is missing, install the repository's documented Python requirements before the event. If the broken record passes, stop: the schema contract has regressed.

</details>

In [166]:
from copy import deepcopy
import json
from pathlib import Path
import jsonschema

root = Path(".")
schema = json.loads((root / "topology-pipeline/inventory.schema.json").read_text())
inventory = json.loads((root / "topology-pipeline/sample-inventory.json").read_text())

broken = deepcopy(inventory)
broken["nodes"][0]["leaf"] = "INVALID LEAF WITH SPACES"

try:
    jsonschema.Draft202012Validator(schema).validate(broken)
    raise AssertionError("broken inventory unexpectedly passed validation")
except jsonschema.ValidationError as error:
    print("EXPECTED REJECTION:", error.message)


EXPECTED REJECTION: 'INVALID LEAF WITH SPACES' does not match '^[a-z0-9]([-a-z0-9]*[a-z0-9])?$'


### Production reconciliation rules

1. Merge authoritative sources with explicit field ownership and precedence.
2. Validate completeness, label syntax, uniqueness, and graph consistency.
3. Publish only a coherent version; never guess through conflicting sources.
4. Report freshness, unmapped nodes, drift, and publish errors.
5. Cordon and drain before changing a running node’s physical-domain identity.
6. Retain last-known-good data only under an explicit staleness policy.

A scheduler can make a perfectly correct decision against incorrect labels. That silent failure mode is why the data contract matters as much as the algorithm.

---
# Phase 6 — Validate topology intent before allocating GPUs

> **Presentation mapping — Live slide 34:** explain CEL for object-local rules and a webhook for external policy. The code exercises are optional if the room is on schedule.

Admission policy rejects malformed intent; it must not choose Nodes or replace Kueue.

- Use **CEL** for fast, object-local invariants.
- Use an external webhook only for richer organizational state or external entitlement.
- Keep slow or fragile DCIM calls off the Kubernetes write path when possible.

## 15 — Inspect the CEL policy

> **Presentation cue — Live slide 34:** show the policy file only if time permits; the required hands-on path already ended at Checkpoint 3.

The policy is deliberately scoped to labeled namespaces and Jobs explicitly marked as distributed training. It requires a queue, exactly one required/preferred annotation, and an approved level.

### Read the API-entry rules for distributed training Jobs

`OPTIONAL`  `READ-ONLY`

**Why this cell is here**

This file is an API-entry checklist. It rejects distributed-training Jobs that forget their Kueue queue, omit topology intent, or request an unapproved topology level.

**What it does**

- Prints the ValidatingAdmissionPolicy match scope, CEL expressions/messages, and its binding.
- Exposes rules requiring a queue, exactly one topology mode, and an approved level for labeled distributed Jobs.

**Expected result**

You can identify scope, validation expressions, failure messages, and binding actions.

<details>
<summary><strong>Implementation details, interpretation and recovery</strong></summary>

**Runs or reads**

- `sed` against `webhook/validating-admission-policy.yaml`.

**State changed:** None; this cell only reads the manifest.

**How to interpret the result:** CEL validates object-local intent at API admission. It does not discover fabric state, reserve Kueue quota, or choose Nodes.

**If it fails:** If the cluster rejects this API kind later, verify the Kubernetes version and feature support; the file inspection itself remains useful.

</details>

In [ ]:
%%bash
sed -n '1,260p' webhook/validating-admission-policy.yaml


### Install the CEL policy and test an invalid Job

`OPTIONAL`  `CHANGES LAB`

**Why this cell is here**

Install the scoped CEL guardrail and prove that malformed distributed-training intent is rejected before persistence.

**What it does**

- Creates or updates cluster-scoped admission policy objects.
- Exercises real API defaulting and validation without creating the Job.

**Expected result**

The policy applies, then the API returns a denial explaining the missing queue/topology contract.

<details>
<summary><strong>Implementation details, interpretation and recovery</strong></summary>

**Runs or reads**

- `make admission-policy` applies the policy and binding.
- `kubectl apply --dry-run=server` sends the unaware Job through API admission; `|| true` preserves notebook flow because denial is expected.

**State changed:** The CEL policy/binding remain installed. The invalid Job is not stored because this is server-side dry-run.

**How to interpret the result:** A rejected dry-run is the green result. The shell's zero exit is forced only so Jupyter does not mark the expected rejection as an execution failure.

**If it fails:** If the Job is accepted, inspect policy binding and namespace/workload labels. If the API kind is unsupported, skip this optional lab and retain the manifest as reference.

</details>

In [ ]:
%%bash
make admission-policy
kubectl apply --dry-run=server -f jobs/training-unaware.yaml || true


### Confirm that a correctly declared Job passes admission

`OPTIONAL`  `READ-ONLY`

**Why this cell is here**

Show that the guardrail is selective: it blocks malformed intent without blocking the valid required-topology Job.

**What it does**

- Sends the object through API schema, admission policy, defaulting, and validation, then discards it.

**Expected result**

The API prints `job.batch/training-topology-aware configured (server dry run)` or equivalent, with no denial.

<details>
<summary><strong>Implementation details, interpretation and recovery</strong></summary>

**Runs or reads**

- `kubectl apply --dry-run=server` for the reviewed strict-locality Job.

**State changed:** No Job is persisted; the previously installed admission policy remains.

**How to interpret the result:** Negative and positive tests together prove policy selectivity. A policy that rejects everything is not a useful guardrail.

**If it fails:** Read the admission message, compare the queue label and exactly-one topology annotation with the CEL expressions, then rerun.

</details>

In [ ]:
%%bash
kubectl apply --dry-run=server -f jobs/training-topology-required.yaml


## 16 — Inspect and test the richer Go policy core

> **Presentation cue — optional extension to Live slide 34:** this corresponds to master slide 52 and is take-home reference material, not a required live demo.

The Go package demonstrates a rule that is easy to get wrong: whole-Job GPU footprint is `GPU per Pod × parallelism`. The package is policy logic only; a production webhook still needs AdmissionReview handling, TLS, authentication, HA, metrics, and rollout controls.

### Inspect and test the advanced Go policy example

`TAKE-HOME`  `READ-ONLY`

**Why this cell is here**

This shows a more advanced policy example written in Go, then runs its tests when Go is available. It is reference logic, not a webhook server running in this lab.

**What it does**

- Shows whole-Job GPU footprint calculation (`per-Pod GPU × parallelism`) and policy validation.
- Runs table-driven unit tests locally, or emits an explicit skip.

**Expected result**

Source prints and tests report `ok .../webhook`, or the documented `SKIP` appears on an attendee machine without Go.

<details>
<summary><strong>Implementation details, interpretation and recovery</strong></summary>

**Runs or reads**

- `sed` for the Go policy core; `go test ./webhook` only when Go is installed.

**State changed:** No Kubernetes state changes. Go may write compiler/module artifacts to its normal local caches.

**How to interpret the result:** The example isolates testable decision logic. AdmissionReview transport, TLS, authentication, high availability, telemetry, and safe rollout remain production responsibilities.

**If it fails:** A skip is acceptable for the 90-minute route. On the instructor machine, run `go test ./webhook -v` to diagnose failures.

</details>

In [ ]:
%%bash
sed -n '1,260p' webhook/topology_validator.go
if command -v go >/dev/null 2>&1; then go test ./webhook; else echo 'SKIP: Go is not installed; run this test on the instructor machine.'; fi


---
# Phase 7 — Prove placement quality with three evidence planes

> **Presentation mapping — Live slides 35–37:** join placement intent, hardware activity, and model progress, then explain the benchmark evidence contract.

| Evidence plane | Examples | What it proves |
|---|---|---|
| Control plane | Workload conditions, topologyAssignment, Pod node, Node labels | Intended and actual placement |
| Hardware/network | DCGM, NVLink/PCIe counters, NIC/fabric telemetry | Device and link activity |
| Application | step time, tokens/s, samples/s, loss, MFU | Useful model progress |

No single plane is sufficient. Stable job/run identifiers and synchronized timestamps are necessary for correlation.

## 17 — Inspect the DCGM reference rules

> **Presentation mapping — Live slides 35–36:** these files support the observability explanation and are not applied to the fake cluster.

These rules are **not applied** to the fake cluster. They are take-home examples for a real GPU environment. DCGM reports activity; it does not directly report application MFU.

### Read the real-GPU monitoring examples

`TAKE-HOME`  `READ-ONLY`

**Why this cell is here**

These files show what you would monitor on real GPU hardware: one contains Prometheus rules for GPU activity, and the other explains how useful model work is calculated. They are not installed in the laptop lab.

**What it does**

- Prints DCGM-derived alert/recording-rule examples.
- Prints the inputs, assumptions, and limitations of application-derived Model FLOP Utilization.

**Expected result**

You can separate SM/NVLink activity metrics from throughput/MFU and identify stable labels needed to correlate a run with placement.

<details>
<summary><strong>Implementation details, interpretation and recovery</strong></summary>

**Runs or reads**

- `sed` against example Prometheus rules and the MFU calculation notes.

**State changed:** None; the rules are deliberately not applied to kind because it has no NVIDIA devices or DCGM exporter.

**How to interpret the result:** DCGM says what hardware is doing; application metrics say whether useful training progresses. Neither alone proves placement quality.

**If it fails:** No live recovery is needed. If adapting to production, validate metric names against your DCGM exporter version before applying any rule.

</details>

In [ ]:
%%bash
sed -n '1,280p' observability/dcgm-prometheus-rules.yaml
sed -n '1,260p' observability/mfu-notes.md


### MFU interpretation

```text
MFU = achieved useful model FLOPs per second
      ──────────────────────────────────────
      precision-matched theoretical peak FLOPs per second
```

Use framework-derived throughput and a model-specific FLOP estimate for the numerator. Match precision and dense/sparse assumptions in the denominator. High SM activity alone can coexist with poor useful progress when ranks wait on communication.

## 18 — Benchmark evidence, not benchmark theatre

> **Presentation mapping — Live slide 37:** inspect the evidence template; do not present the analytical scenario as measured H100 data.

A defensible comparison holds constant the model, framework, precision, batch, GPU count, software stack, warm-up, sample count, and fabric health. Vary the placement domain and collect at least:

- NCCL algorithm and bus bandwidth;
- step time and tokens/s;
- application-derived MFU;
- topology assignment and actual node/leaf identity;
- queue delay and run duration.

### Read the benchmark procedure and results template

`TAKE-HOME`  `READ-ONLY`

**Why this cell is here**

The Markdown file is a checklist for running a fair benchmark. The CSV is the empty form where measured results should be recorded.

**What it does**

- Shows controlled variables, repetitions, warm-up, placement identity, NCCL/application metrics, and required provenance.
- Shows the CSV columns accepted by the summarizer.

**Expected result**

The protocol distinguishes measured evidence from analytical teaching data and the CSV has fields for placement, speed, MFU, and source.

<details>
<summary><strong>Implementation details, interpretation and recovery</strong></summary>

**Runs or reads**

- `sed` against the benchmark protocol and empty measured-results template.

**State changed:** None.

**How to interpret the result:** A screenshot of faster Pods is not a benchmark. Comparable runs must hold workload, hardware, and software constant and preserve source metadata.

**If it fails:** If collecting real data, copy the template rather than overwriting it, mark `source=measured`, and retain raw logs alongside the summary.

</details>

In [ ]:
%%bash
sed -n '1,260p' benchmarks/benchmark-template.md
sed -n '1,80p' benchmarks/results-template.csv


### Calculate the analytical placement comparison

`LIVE`  `READ-ONLY`

**Why this cell is here**

Demonstrate how placement comparisons are normalized while keeping scenario values clearly separated from measurements.

**What it does**

- Validates required columns and positive numeric values.
- Uses the first row as baseline, then calculates relative efficiency, throughput degradation, GPU-hour multiplier, and displays MFU.

**Expected result**

A placement comparison table prints with efficiency, degradation, GPU-hour multiplier, and MFU columns.

<details>
<summary><strong>Implementation details, interpretation and recovery</strong></summary>

**Runs or reads**

- `benchmarks/summarize_results.py` with `scenario-model.csv` and the explicit `--allow-analytical` acknowledgement.

**State changed:** None; it reads a CSV and prints a table.

**How to interpret the result:** These are arithmetic demonstrations, not H100 measurements. Remove the flag and the script intentionally refuses non-measured input.

**If it fails:** If validation fails, fix the named CSV field. Never relabel scenario rows as measured merely to bypass the evidence guard.

</details>

In [ ]:
%%bash
python3 benchmarks/summarize_results.py benchmarks/scenario-model.csv --allow-analytical


**Interpretation:** the 40–60% degradation range in the session premise is workload-, hardware-, collective-, and topology-dependent. The fake lab cannot validate it. Replace the results template with measured `nccl-tests` and training data before making a production claim.

---
# Phase 8 — Validate, recover, and take the architecture home

> **Presentation mapping — Live slides 38–39:** use the architecture to connect all owners, finish with the five rules, and leave the cluster at Checkpoint 3.

## Failure map

| Failure | Symptom | First response |
|---|---|---|
| Missing host dependency | Preflight failure | Fix before the workshop |
| Missing labels or allocatable GPU | Checkpoint 1 verification fails | `make checkpoint-1` |
| Leaked pause Pods | Later Pods remain Pending | `make lab1-reset` |
| Kueue controller/config drift | ClusterQueue inactive | `make checkpoint-2` |
| Stale Lab 3 workloads | Unexpected quota/topology state | `make checkpoint-3` |
| Wrong Kubernetes context | Safety check fails | Switch explicitly to `kind-topology-lab` |

The scripts refuse mutations outside the named lab context.

## 19 — Final offline validation

> **Presentation cue — after Live slide 39:** this is a take-home validation step, not a command to run while the audience is waiting.

### Run the final offline repository checks

`TAKE-HOME`  `READ-ONLY`

**Why this cell is here**

Confirm that exploration did not leave edited workshop assets in an invalid or inconsistent state.

**What it does**

- Revalidates repository artifacts; it does not inspect the live cluster.

**Expected result**

Static validation and all unit tests pass.

<details>
<summary><strong>Implementation details, interpretation and recovery</strong></summary>

**Runs or reads**

- The same static shell/YAML/schema/notebook checks and Python unit tests explained in the pre-event section.

**State changed:** None.

**How to interpret the result:** The checked-in teaching materials remain coherent. Use `make checkpoint-3` separately to validate or recover live cluster state.

**If it fails:** Return to the first reported failing file or test. Generated notebooks must be rebuilt from `scripts/build-notebook.js`, not hand-edited.

</details>

In [ ]:
%%bash
make test


<details>
<summary><strong>Optional instructor integration test</strong></summary>

`make smoke` creates and deletes only the kind cluster named `topology-lab`, then executes every live lab. Run it before the event on the instructor machine—not during the participant session.

```bash
make smoke
```
</details>

## Production-readiness checklist

- [ ] Every topology label has an authoritative owner and freshness SLO.
- [ ] Domain definitions match the actual network failure/performance boundaries.
- [ ] Required and preferred policies are tied to workload classes and deadlines.
- [ ] Admission rejects missing, conflicting, or unauthorized topology intent.
- [ ] Queue quota and physical domain capacity are monitored independently.
- [ ] Node-local CPU, memory, GPU, and NIC alignment is configured and verified.
- [ ] Workload status, Pod placement, fabric telemetry, and application metrics can be correlated.
- [ ] Benchmark claims carry configuration, placement identity, repetitions, and source.
- [ ] Recovery paths are idempotent, bounded, and rehearsed.

> **Placement quality is a resource.**

## Cleanup

> **Presentation cue — after Live slide 39:** run this only when participants no longer need the reference cluster.

This removes only the kind cluster named `topology-lab`. Cached manifests and images remain available for recovery or another workshop run.

### Delete the workshop cluster when you are finished

`OPTIONAL`  `DELETES LAB`

**Why this cell is here**

Release the Docker containers and storage owned by the workshop cluster after participants no longer need it.

**What it does**

- Deletes only the named kind cluster and its node containers/network.

**Expected result**

Ends with `cluster removed; preflight cache preserved for recovery`; `kind get clusters` no longer lists `topology-lab`.

<details>
<summary><strong>Implementation details, interpretation and recovery</strong></summary>

**Runs or reads**

- `Makefile` → `scripts/cleanup.sh` → `kind delete cluster --name topology-lab`.

**State changed:** Removes Kubernetes state inside `topology-lab`. It preserves `.workshop-cache/` and Docker images for faster recreation.

**How to interpret the result:** The live lab is gone, but `make checkpoint-1` can recreate it offline from cached inputs.

**If it fails:** Deletion cannot restore in-cluster objects. Recreate the known lab state with `make checkpoint-1` or `make checkpoint-2` for Kueue/TAS.

**Safety / caveat:** Do not run this before the final demonstrations. The script is intentionally hard-coded to the workshop cluster name.

</details>

In [141]:
%%bash
make clean


bash scripts/cleanup.sh
==> Deleting only kind cluster 'topology-lab'


Deleting cluster "topology-lab" ...
Deleted nodes: ["topology-lab-control-plane" "topology-lab-worker" "topology-lab-worker3" "topology-lab-worker2" "topology-lab-worker4"]


OK  cluster removed; preflight cache preserved for recovery


## Take-home references in this repository

- `README.md` — authoritative live path and recovery contract
- `nvlink-workshop-tutorial.md` — full curriculum and technical narrative
- `topology-pipeline/` — schema, sample inventory, and production data-contract guidance
- `webhook/` — CEL policy and unit-tested Go policy core
- `observability/` — DCGM and MFU guidance for real GPU clusters
- `benchmarks/` — measured-results template and analytical example
- `tests/` — offline validation and destructive named-cluster smoke test

The notebook teaches the path; the repository preserves the complete implementation.

In [142]:
%%bash
kind get clusters

demo
kueue-demo
